# UB disengagement reason categorization

Classifies the free-text note in **`Reason for Disengagement:`** (the column *with* the colon) of the PB 4108 disengagement export, for tickets dated **on or before 31 Mar 2024**.

The pipeline: **discover first, classify second.**

| Phase | What it does | Checkpoint (in `artifacts/`) |
|---|---|---|
| 0 | Config, logging, helpers | `run.log` |
| 1 | Load, profile, EDA (n-grams, note types by year, form-value conflicts) | `01_profile.json`, `01_loaded.pkl` |
| 2 | Conservative cleaning in new columns, BLANK vs VALID_TEXT | `02_clean.pkl` |
| 3 | Sentence embeddings, one per unique cleaned note, cached | `embeddings/<model>.npz` |
| 4 | BERTopic (UMAP + HDBSCAN) across several `min_cluster_size` values | `04_clusters_mcs<N>.parquet` |
| 5 | Cluster summaries and a separate outlier (-1) analysis | `05_cluster_cards.json` |
| 6 | LLM taxonomy discovery, then consolidation → proposed taxonomy for **human review** | `llm/`, `taxonomy/` |
| 7 | Per-ticket classification against the **approved** taxonomy | `llm/classification/` |
| 8 | Validation sample and metrics | `outputs/` |
| 9 | Excel workbook | `outputs/` |

**LLM backend.** No API key is used. The LLM steps run through a *file exchange*: the notebook writes a request file (prompt + payload) to `artifacts/llm/`, Claude (in the Claude Code session) writes the matching response file, and the notebook validates it against a JSON schema. To switch to an API later, replace `llm_exchange()` in Phase 0. Nothing else changes.

**Privacy.** Notes name staff and clients, so everything except this notebook is gitignored (see `.gitignore`). The notebook's printed outputs also show notes.

In [ ]:
import os, re, json, html, hashlib, unicodedata, logging, datetime as dt, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

NB_DIR = Path.cwd()
if not (NB_DIR / "ub_reason_categorization.ipynb").exists():
    NB_DIR = Path(r"C:\Users\Alfonso(FonsoP)Panli\repos\cloudstaff_dev\reference_code\UB Reason categorizing")

CONFIG = {
    # --- input ---------------------------------------------------------------
    "input_path": r"C:\Users\Alfonso(FonsoP)Panli\OneDrive - Cloudstaff Pty Ltd\Desktop\Projects\Churn Analysis\Data\PB4108 Disengagement 2020 onwards.xlsx",
    "input_type": "excel",              # "excel" or "csv"
    "sheet_name": "PB 4108",
    "id_col": "Ticket no.",
    "text_col": "Reason for Disengagement:",   # free-text note (WITH colon); the no-colon column is the form dropdown
    "date_col": "Date Disengagement",
    "date_max": "2024-03-31",           # inclusive; None = no filter
    "duplicate_id_handling": "suffix",  # "suffix" (keep all, add __2, __3) | "keep_first" | "error"
    "form_values": ["Voluntary", "Involuntary", "For Record Purpose Only", "Deleted"],  # dropdown values that show up as notes
    "short_note_words": 2,              # notes with <= this many words count as "very short"
    # --- outputs -------------------------------------------------------------
    "artifact_dir": str(NB_DIR / "artifacts"),
    "taxonomy_dir": str(NB_DIR / "taxonomy"),
    "output_dir": str(NB_DIR / "outputs"),
    "output_workbook": "ub_reason_classification.xlsx",
    # --- embeddings ----------------------------------------------------------
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_batch_size": 64,
    # --- clustering ----------------------------------------------------------
    "umap": dict(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42),
    "hdbscan": dict(min_samples=5, metric="euclidean", cluster_selection_method="eom", prediction_data=True),
    "min_cluster_sizes": [10, 20, 30, 50],
    "selected_min_cluster_size": 10,    # chosen after reviewing the comparison in Phase 4
    "outlier_group_distance": 0.35,     # cosine distance for grouping outliers into mini-themes
    # --- LLM -----------------------------------------------------------------
    "llm_backend": "file_exchange",     # Claude in the Claude Code session writes the response files
    "llm_model": "claude-opus-5-5 (in-session, file exchange)",
    "taxonomy_prompt_version": "tax-v1",
    "classification_prompt_version": "cls-v1",
    "classification_batch_size": 150,
    "confidence_threshold": 0.75,
    "validation_sample_size": 500,
    # reviewer scoring rules (validation "Accuracy (reviewer rules)"): agreed with the reviewer 2026-11-25
    "validation_equivalent_pairs": [["ATTENDANCE_AWOL", "SERVICE_DISSATISFACTION"]],  # both true: right if one is primary and the other secondary
    "validation_rollup": {"ROLE_NO_LONGER_NEEDED": "BUSINESS_REASON"},                  # counted as business reason at large
    "spotcheck_size": 100,
    # reviewer's main categories (2026-11-25): every approved category_id must appear here
    "main_category_map": {
        "SERVICE_DISSATISFACTION": "Service Dissatisfaction", "ATTENDANCE_AWOL": "Service Dissatisfaction", "MISCONDUCT": "Service Dissatisfaction",
        "RESIGNATION": "Resignation", "STAFF_MEDICAL": "Resignation",
        "BUSINESS_REASON": "Business Reason", "BUSINESS_DOWNTURN": "Business Reason", "ROLE_NO_LONGER_NEEDED": "Business Reason",
        "ROLE_MOVED_ONSHORE_OR_PROVIDER": "Business Reason", "CLIENT_CLOSURE_OR_ACQUISITION": "Business Reason",
        "CLIENT_ACCOUNT_EXIT": "Business Reason", "CLIENT_BUYOUT": "Business Reason", "NON_PAYMENT": "Business Reason",
        "FOR_RECORD_PURPOSE_ONLY": "For Record", "FORM_DIRECTION_ONLY": "For Record",
        "BLANK": "Other/Unknown", "DELETED": "Other/Unknown", "UNKNOWN": "Other/Unknown",
    },              # fresh blind sample, notes never in any earlier validation file
    "random_seed": 42,
}

ART = Path(CONFIG["artifact_dir"]); TAX = Path(CONFIG["taxonomy_dir"]); OUT = Path(CONFIG["output_dir"])
LLM_DIR = ART / "llm"
for p in [ART, TAX, OUT, LLM_DIR, ART / "embeddings", LLM_DIR / "classification"]:
    p.mkdir(parents=True, exist_ok=True)

# logging: file + notebook, never secrets
logger = logging.getLogger("ub_reason")
logger.setLevel(logging.INFO)
if not logger.handlers:
    fh = logging.FileHandler(ART / "run.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    logger.addHandler(fh)
RUN = {"started": dt.datetime.now().isoformat(timespec="seconds"), "warnings": [], "errors": [], "status": {}}
def warn(msg):
    RUN["warnings"].append(msg); logger.warning(msg); print("WARNING:", msg)
logger.info("Run started")
print("Notebook dir:", NB_DIR)

In [ ]:
# ---- LLM file exchange --------------------------------------------------------
# request:  artifacts/llm/<name>.request.json   (prompt + payload, written by the notebook)
# response: artifacts/llm/<name>.response.json  (written by the LLM, i.e. Claude in-session)
# Swap this function for an API call to automate; the rest of the notebook only sees the parsed JSON.
import jsonschema

class LLMResponsePending(Exception):
    pass

def llm_exchange(name, prompt, payload, schema=None, subdir=None):
    d = LLM_DIR / subdir if subdir else LLM_DIR
    d.mkdir(parents=True, exist_ok=True)
    req = d / f"{name}.request.json"; resp = d / f"{name}.response.json"
    request = {"name": name, "model": CONFIG["llm_model"], "prompt": prompt, "payload": payload}
    req.write_text(json.dumps(request, indent=1, ensure_ascii=False), encoding="utf-8")
    if not resp.exists():
        logger.info(f"LLM request written, awaiting response: {req.name}")
        raise LLMResponsePending(f"Waiting for {resp}")
    try:
        data = json.loads(resp.read_text(encoding="utf-8"))
    except json.JSONDecodeError as e:
        RUN["errors"].append(f"{resp.name}: invalid JSON ({e})"); logger.error(f"invalid JSON {resp.name}")
        raise
    if schema is not None:
        jsonschema.validate(data, schema)   # raises on invalid; never silently accepted
    logger.info(f"LLM response loaded: {resp.name}")
    return data

def sha(s, n=16):
    return hashlib.sha1(str(s).encode("utf-8")).hexdigest()[:n]

## Phase 1 · Load and profile

In [ ]:
def load_input(cfg):
    p = Path(cfg["input_path"])
    if not p.exists():
        raise FileNotFoundError(f"Input not found: {p}")
    if cfg["input_type"] == "excel":
        df = pd.read_excel(p, sheet_name=cfg["sheet_name"])
    elif cfg["input_type"] == "csv":
        df = pd.read_csv(p)
    else:
        raise ValueError("input_type must be 'excel' or 'csv'")
    missing = [c for c in (cfg["id_col"], cfg["text_col"]) if c not in df.columns]
    if missing:
        raise KeyError(f"Configured columns missing from input: {missing}")
    df.insert(0, "source_row", df.index + 2)          # Excel row number in the source sheet
    if cfg.get("date_max"):
        dates = pd.to_datetime(df[cfg["date_col"]], errors="coerce")
        n_bad = dates.isna().sum()
        if n_bad: warn(f"{n_bad} rows have an unparseable {cfg['date_col']} and are excluded by the date filter")
        df = df[dates <= pd.Timestamp(cfg["date_max"])].copy()
    return df.reset_index(drop=True)

def assign_record_ids(df, cfg):
    ids = df[cfg["id_col"]].astype("string")
    if ids.isna().any():
        warn(f"{ids.isna().sum()} rows have no {cfg['id_col']}; given ROW-<source_row> ids")
        ids = ids.fillna("ROW-" + df["source_row"].astype(str))
    dup = ids.duplicated(keep=False)
    if dup.any():
        warn(f"{dup.sum()} rows share a duplicated {cfg['id_col']}: {sorted(ids[dup].unique())}")
        if cfg["duplicate_id_handling"] == "error":
            raise ValueError("Duplicate record IDs")
        if cfg["duplicate_id_handling"] == "keep_first":
            keep = ~ids.duplicated(keep="first"); df, ids = df[keep].copy(), ids[keep]
        else:
            n = ids.groupby(ids).cumcount() + 1
            ids = ids.where(n == 1, ids + "__" + n.astype(str))
    df.insert(0, "record_id", ids.values)
    return df, dup

df = load_input(CONFIG)
df, dup_id_mask = assign_record_ids(df, CONFIG)
TEXT = CONFIG["text_col"]
print(f"Loaded {len(df):,} rows x {df.shape[1]} cols ({CONFIG['date_col']} <= {CONFIG['date_max']})")
display(df.loc[dup_id_mask, ["record_id", CONFIG["id_col"], "Engagement ID", "Client ShortCode", CONFIG["date_col"], TEXT]])

In [ ]:
def profile_notes(df, text_col, short_words):
    raw = df[text_col]
    stripped = raw.astype("string").str.strip()
    is_blank = stripped.isna() | (stripped == "")
    nb = stripped[~is_blank]
    words = nb.str.split().str.len()
    vc = nb.value_counts()
    n = len(df)
    rows = [
        ("Total records", n),
        ("Blank notes", int(is_blank.sum())),
        ("Non-blank notes", int((~is_blank).sum())),
        ("Unique non-blank notes", int(nb.nunique())),
        ("Records whose note is repeated elsewhere", int(nb.duplicated(keep=False).sum())),
        ("Records with a note seen only once", int((vc == 1).sum())),
        (f"Very short notes (<= {short_words} words)", int((words <= short_words).sum())),
        ("Notes over 50 words", int((words > 50).sum())),
    ]
    prof = pd.DataFrame(rows, columns=["metric", "records"])
    prof["pct_of_all_records"] = (prof["records"] / n * 100).round(1)
    length = words.describe(percentiles=[.25, .5, .75, .9, .99]).round(1)
    longest = (df.loc[~is_blank, ["record_id", text_col]].assign(words=words)
               .sort_values("words", ascending=False).head(10))
    top = vc.head(25).rename_axis("raw_note").reset_index(name="records")
    top["pct_of_all_records"] = (top["records"] / n * 100).round(1)
    return prof, length, longest, top

profile, note_length, longest_notes, top_raw_notes = profile_notes(df, TEXT, CONFIG["short_note_words"])
display(profile)
print("Words per non-blank note:"); display(note_length.to_frame("words").T)
display(top_raw_notes)

In [ ]:
print("Longest notes (often pasted client emails):")
display(longest_notes.assign(preview=longest_notes[TEXT].str.slice(0, 160))[["record_id", "words", "preview"]])

(ART / "01_profile.json").write_text(json.dumps({
    "profile": profile.to_dict("records"), "note_length": note_length.to_dict(),
    "top_raw_notes": top_raw_notes.to_dict("records"),
    "duplicate_ids": df.loc[dup_id_mask, "record_id"].tolist()}, indent=1, default=str), encoding="utf-8")
df.to_pickle(ART / "01_loaded.pkl")   # pickle keeps mixed-type source columns exactly
RUN["status"]["phase1"] = "done"; logger.info(f"Phase 1: {len(df)} records loaded")

### EDA 1 · What kind of text is in the note field, by year?

The note field holds three different things: real free text, a **form value** copied in (`Voluntary`, `Involuntary`, `For Record Purpose Only`), or nothing. The mix changes sharply by year, so free-text notes describe 2020–22 much better than 2023–24.

In [ ]:
fv = {v.lower() for v in CONFIG["form_values"]}
s = df[TEXT].astype("string").str.strip()
df["note_kind"] = np.where(s.isna() | (s == ""), "BLANK",
                  np.where(s.str.lower().isin(fv), "FORM VALUE", "FREE TEXT"))
kind_year = pd.crosstab(df["Year Disengagement"], df["note_kind"], margins=True, margins_name="Total")
display(kind_year)
display((pd.crosstab(df["Year Disengagement"], df["note_kind"], normalize="index") * 100).round(0).astype(int).add_suffix(" %"))

### EDA 2 · Most common 2–4-word phrases (free-text notes only)

Counted once per ticket. Form values are left out because they would dominate. Stop words are kept inside phrases ("no longer needed" needs them), but phrases that start or end with a stop word are dropped.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
NEGATIONS = {"no", "not", "never", "nor", "none", "cannot"}
STOP = set(ENGLISH_STOP_WORDS) - NEGATIONS

def top_ngrams(texts, n, k=15):
    cv = CountVectorizer(ngram_range=(n, n), binary=True, lowercase=True, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b")
    X = cv.fit_transform(texts)
    counts = pd.Series(np.asarray(X.sum(0)).ravel(), index=cv.get_feature_names_out())
    keep = [g for g in counts.index if g.split()[0] not in STOP and g.split()[-1] not in STOP]
    return counts[keep].sort_values(ascending=False).head(k)

free = df.loc[df["note_kind"] == "FREE TEXT", TEXT].astype(str).str.replace("_x000D_", " ")
ngram_tables = {n: top_ngrams(free, n) for n in (2, 3, 4)}
display(pd.concat({f"{n}-grams": t.rename_axis("phrase").reset_index(name="tickets") for n, t in ngram_tables.items()}, axis=1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
for ax, (n, t) in zip(axes, ngram_tables.items()):
    t = t.iloc[::-1]
    ax.barh(t.index, t.values, color="#3B6FB6", height=0.7)
    ax.set_title(f"Top {n}-word phrases (tickets)", loc="left", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False); ax.tick_params(labelsize=8)
    ax.grid(axis="x", color="#e5e5e5"); ax.set_axisbelow(True)
plt.show()

### EDA 3 · Do the `Voluntary` / `Involuntary` notes agree with the form?

These notes are the direction dropdown copied into the note field (2022+). Checked against the structured fields:
* the dropdown column `Reason for Disengagement` (no colon),
* the sub-reason fields (`Type of Voluntary Reason`, `Type of Involuntary`, `Type of Customer Initiated`, `Reason for Resignation`),
* the workbook's derived `Master Reason`.

The free-text notes are also checked: notes whose *wording* is clearly a resignation, or clearly client / business / performance, against the dropdown direction.

In [ ]:
ncl = df[TEXT].astype("string").str.strip().fillna("<blank>")
grp = ncl.where(ncl.isin(["Voluntary", "Involuntary", "For Record Purpose Only", "<blank>"]), "<free text>")
print("Note value vs dropdown direction"); display(pd.crosstab(grp, df["Reason for Disengagement"], margins=True))
print("Note value vs derived Master Reason"); display(pd.crosstab(grp, df["Master Reason"], margins=True))

vol, inv = df[ncl == "Voluntary"], df[ncl == "Involuntary"]
inv_fields = ["Type of Involuntary", "Type of Customer Initiated", "Type of Cloudstaff Initiated"]
vol_conf = vol[vol[inv_fields].notna().any(axis=1)]
print(f"'Voluntary' notes that also carry involuntary sub-reasons: {len(vol_conf)} of {len(vol)}")
display(pd.crosstab(vol_conf["Type of Customer Initiated"].fillna("(none)"), vol_conf["Master Reason"]))
real_vol_reason = inv["Reason for Resignation"].notna() | (inv["Type of Voluntary Reason"].notna() & ~inv["Type of Voluntary Reason"].astype(str).str.startswith("NA."))
print(f"'Involuntary' notes that carry a resignation reason: {real_vol_reason.sum()} of {len(inv)}")
display(inv.loc[real_vol_reason, "Reason for Resignation"].value_counts(dropna=False).head(8).to_frame("tickets"))

In [ ]:
ft = grp == "<free text>"
note = df[TEXT].astype(str)
resign_words = note.str.contains("resign", case=False) & ~note.str.contains("client|business|pull", case=False)
client_words = (note.str.contains(r"performance|pull ?out|business|role no longer|downturn|slow ?down|ncns|awol|attendance", case=False)
                & ~note.str.contains("resign", case=False))
conf = pd.DataFrame({
    "free-text wording": ["clearly resignation", "clearly client / business / performance"],
    "tickets": [int((ft & resign_words).sum()), int((ft & client_words).sum())],
    "dropdown = Voluntary": [int((ft & resign_words & (df["Reason for Disengagement"] == "Voluntary")).sum()),
                             int((ft & client_words & (df["Reason for Disengagement"] == "Voluntary")).sum())],
    "dropdown = Involuntary": [int((ft & resign_words & (df["Reason for Disengagement"] == "Involuntary")).sum()),
                               int((ft & client_words & (df["Reason for Disengagement"] == "Involuntary")).sum())],
})
display(conf)
print("Examples: client/business/performance wording but dropdown = Voluntary (so Master Reason = Resignation)")
display(df.loc[ft & client_words & (df["Reason for Disengagement"] == "Voluntary"), ["record_id", TEXT, "Master Reason"]]
        .assign(**{TEXT: lambda x: x[TEXT].str.slice(0, 130)}).head(10))

**Takeaways**
* A `Voluntary` / `Involuntary` note carries **direction only, no reason**. It matches the dropdown in almost every case. The disagreements are *inside the structured fields*: 63 `Voluntary` tickets also carry involuntary sub-reasons (mostly *Customer Initiated → Business Reasons / Service Dissatisfaction*), but `Master Reason` still calls them Resignation.
* Free text clearly about the client / business / performance sometimes sits under a `Voluntary` dropdown. Those tickets inherit `Master Reason = Resignation`.
* So the note classification below uses **the note text only**. It is compared with `Master Reason` afterwards (Phase 9) to surface disagreements, instead of copying the form.

## Phase 2 · Preprocess (conservative)

The original note is untouched. `note_clean` is new. There's no stop-word removal, stemming or lower-casing before embedding, and negations are kept. A note is **BLANK** if it's empty, whitespace only, a template placeholder (`{{FIELD_…}}`) or has no letters or digits. Everything else is **VALID_TEXT**, including junk like "test", which the classifier can still call BLANK.

In [ ]:
HTML_TAG = re.compile(r"</?[a-zA-Z][a-zA-Z0-9]*(\s[^<>]*)?/?>")
PLACEHOLDER = re.compile(r"^\{\{[^{}]*\}\}$")
QUOTES = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-", "\u00a0": " "}

def clean_note(x):
    if x is None or (not isinstance(x, str) and pd.isna(x)):
        return ""
    s = str(x).replace("_x000D_", " ")
    s = html.unescape(s)
    s = HTML_TAG.sub(" ", s)
    s = unicodedata.normalize("NFKC", s)
    for a, b in QUOTES.items():
        s = s.replace(a, b)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def processing_status(clean):
    if clean == "" or PLACEHOLDER.match(clean) or not re.search(r"[A-Za-z0-9]", clean):
        return "BLANK"
    return "VALID_TEXT"

df["note_clean"] = df[TEXT].map(clean_note)
df["processing_status"] = df["note_clean"].map(processing_status)
df["note_key"] = np.where(df["processing_status"] == "VALID_TEXT", df["note_clean"].map(sha), "")
df.to_pickle(ART / "02_clean.pkl")
display(df["processing_status"].value_counts().to_frame("records"))
print("Cleaned notes that became BLANK although the raw cell had text:")
display(df.loc[(df["processing_status"] == "BLANK") & df[TEXT].notna(), TEXT].value_counts().to_frame("records"))
RUN["status"]["phase2"] = "done"

## Phase 3 · Sentence embeddings

One embedding per **unique cleaned note**. `unique_notes` maps each note back to its records (`records` = how many tickets share it). Embeddings are cached in `artifacts/embeddings/` keyed by note hash, so reruns only embed new notes.

In [ ]:
from sentence_transformers import SentenceTransformer

valid = df[df["processing_status"] == "VALID_TEXT"]
unique_notes = (valid.groupby("note_key")
    .agg(note_clean=("note_clean", "first"), records=("record_id", "size"),
         note_original=(TEXT, lambda s: s.value_counts().index[0]))   # most common raw wording
    .sort_values("records", ascending=False).reset_index())
assert unique_notes["records"].sum() == len(valid)

def get_embeddings(texts, keys, model_name):
    cache = ART / "embeddings" / (model_name.replace("/", "__") + ".npz")
    have = {}
    if cache.exists():
        z = np.load(cache, allow_pickle=False)
        have = dict(zip(z["keys"].tolist(), z["vectors"]))
    todo = [(k, t) for k, t in zip(keys, texts) if k not in have]
    if todo:
        model = SentenceTransformer(model_name)
        vecs = model.encode([t for _, t in todo], batch_size=CONFIG["embedding_batch_size"],
                            normalize_embeddings=True, show_progress_bar=False)
        have.update({k: v for (k, _), v in zip(todo, vecs)})
        np.savez(cache, keys=np.array(list(have.keys())), vectors=np.vstack(list(have.values())))
    logger.info(f"Embeddings: {len(keys)} needed, {len(todo)} newly computed ({model_name})")
    print(f"{len(keys):,} unique notes; {len(todo):,} newly embedded, {len(keys)-len(todo):,} from cache")
    return np.vstack([have[k] for k in keys])

E = get_embeddings(unique_notes["note_clean"].tolist(), unique_notes["note_key"].tolist(), CONFIG["embedding_model"])
print("Embedding matrix:", E.shape)
RUN["status"]["phase3"] = "done"

## Phase 4 · Semantic clusters (BERTopic = UMAP → HDBSCAN → c-TF-IDF keywords)

Clusters are fitted on unique notes. **Sizes are reported in tickets** (the `records` each note represents). Noise stays as cluster **-1** and is never forced into a cluster. UMAP has a fixed seed, so runs repeat.

Note: HDBSCAN can't weight points, so a note repeated 1,000 times counts once when the clusters form.

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.metrics import silhouette_score

KEYWORD_STOP = sorted(STOP)   # keyword extraction only; embeddings saw the full text

def run_bertopic(docs, emb, min_cluster_size):
    model = BERTopic(
        embedding_model=None,
        umap_model=UMAP(**CONFIG["umap"]),
        hdbscan_model=HDBSCAN(min_cluster_size=min_cluster_size, **CONFIG["hdbscan"]),
        vectorizer_model=CountVectorizer(ngram_range=(1, 3), stop_words=KEYWORD_STOP, min_df=2),
        calculate_probabilities=False, verbose=False)
    topics, probs = model.fit_transform(docs, embeddings=emb)
    return model, np.asarray(topics), np.asarray(probs)

def cluster_quality(labels, emb, weights):
    rows = []
    for c in sorted(set(labels) - {-1}):
        m = labels == c
        cen = emb[m].mean(0); cen /= np.linalg.norm(cen)
        rows.append((c, m.sum(), weights[m].sum(), float((emb[m] @ cen).mean())))
    return pd.DataFrame(rows, columns=["cluster", "unique", "records", "coherence"])

docs = unique_notes["note_clean"].tolist(); w = unique_notes["records"].to_numpy()
experiments, models = [], {}
for mcs in CONFIG["min_cluster_sizes"]:
    path = ART / f"04_clusters_mcs{mcs}.parquet"
    model, topics, probs = run_bertopic(docs, E, mcs)
    models[mcs] = model
    pd.DataFrame({"note_key": unique_notes["note_key"], "cluster_id": topics, "probability": probs}).to_parquet(path, index=False)
    q = cluster_quality(topics, E, w); noise = topics == -1
    sil = silhouette_score(E[~noise], topics[~noise], metric="cosine") if (~noise).sum() > 2 and len(q) > 1 else np.nan
    experiments.append({
        "min_cluster_size": mcs, "clusters": len(q),
        "outlier_unique_%": round(noise.mean() * 100, 1),
        "outlier_records_%": round(w[noise].sum() / w.sum() * 100, 1),
        "largest_cluster_records": int(q["records"].max()), "median_cluster_records": int(q["records"].median()),
        "mean_coherence": round(q["coherence"].mean(), 3), "silhouette_cosine": round(sil, 3)})
    logger.info(f"BERTopic mcs={mcs}: {len(q)} clusters, {noise.sum()} outlier notes")
experiments = pd.DataFrame(experiments)
display(experiments)

In [ ]:
# Side by side: what each configuration calls its biggest clusters (top 4 keywords), to check merging and splitting by eye
def top_clusters(mcs, k=12):
    lab = pd.read_parquet(ART / f"04_clusters_mcs{mcs}.parquet")["cluster_id"].to_numpy()
    q = cluster_quality(lab, E, w).sort_values("records", ascending=False).head(k)
    kw = [", ".join(t for t, _ in models[mcs].get_topic(c)[:4]) for c in q["cluster"]]
    return pd.Series([f"{r} | {k}" for r, k in zip(q["records"], kw)], name=f"mcs={mcs}")
display(pd.concat([top_clusters(m).reset_index(drop=True) for m in CONFIG["min_cluster_sizes"]], axis=1))

**Choosing a configuration.** The choice isn't made by any single metric. Larger `min_cluster_size` merges distinct reasons into a few big clusters, for example resignation sub-reasons, office/WFH moves and business slowdowns. The smallest setting keeps those reasons apart while the clusters stay coherent, and its outliers are reviewed separately in Phase 5 anyway. **`selected_min_cluster_size` is set in CONFIG.**

In [ ]:
SEL = CONFIG["selected_min_cluster_size"]
assign = pd.read_parquet(ART / f"04_clusters_mcs{SEL}.parquet")
unique_notes = unique_notes.drop(columns=["cluster_id", "probability"], errors="ignore").merge(assign, on="note_key", how="left")
topic_model = models[SEL]
labels = unique_notes["cluster_id"].to_numpy()
df = df.drop(columns=["cluster_id"], errors="ignore").merge(unique_notes[["note_key", "cluster_id"]], on="note_key", how="left")
assert len(df) == len(valid) + (df["processing_status"] == "BLANK").sum()
print(f"Selected min_cluster_size={SEL}: {len(set(labels)) - 1} clusters; outliers = {(labels == -1).sum()} notes / {w[labels == -1].sum()} tickets")
RUN["status"]["phase4"] = f"done (mcs={SEL})"

## Phase 5 · Cluster summaries and outlier analysis

For each cluster, the summary shows sizes, c-TF-IDF keywords, mean within-cluster similarity and mean HDBSCAN membership probability. It also lists:
* **representative notes**: chosen for diversity (maximal marginal relevance), starting from the most common note;
* the notes **closest to** and **furthest from** the centroid.

Notes are shown in their original wording.

Outliers (-1) are grouped into mini-themes by cosine distance, so small recurring reasons aren't lost.

In [ ]:
def mmr_pick(idx, emb, weights, k=8, lam=0.6):
    idx = list(idx)
    if len(idx) <= k: return idx
    cen = emb[idx].mean(0)
    rel = emb[idx] @ cen + 0.02 * np.log1p(weights[idx])
    chosen = [idx[int(np.argmax(weights[idx]))]]
    while len(chosen) < k:
        best, best_s = None, -1e9
        for j, i in enumerate(idx):
            if i in chosen: continue
            s = lam * rel[j] - (1 - lam) * max(float(emb[i] @ emb[c]) for c in chosen)
            if s > best_s: best, best_s = i, s
        chosen.append(best)
    return chosen

def fmt(i):
    r = unique_notes.iloc[i]
    return f"[{r.records}] {r.note_original.replace('_x000D_', ' ').strip()[:300]}"

def cluster_card(cid, idx, kind="cluster", keywords=None):
    idx = np.asarray(idx)
    sub = E[idx]; cen = sub.mean(0); cen /= np.linalg.norm(cen)
    sim = sub @ cen; order = idx[np.argsort(-sim)]
    pair = (sub @ sub.T)[np.triu_indices(len(idx), 1)] if len(idx) > 1 else np.array([1.0])
    probs = unique_notes.loc[idx, "probability"]
    return {
        "cluster_id": str(cid), "kind": kind,
        "unique_notes": int(len(idx)), "records": int(w[idx].sum()),
        "keywords": keywords or [],
        "mean_within_similarity": round(float(pair.mean()), 3),
        "mean_membership_probability": round(float(probs.mean()), 3) if kind == "cluster" else None,
        "representative_notes": [fmt(i) for i in mmr_pick(idx, E, w)],
        "closest_to_centroid": [fmt(i) for i in order[:5]],
        "furthest_from_centroid": [fmt(i) for i in order[::-1][:4]] if len(idx) > 5 else [],
    }

cards = []
for c in sorted(set(labels) - {-1}):
    idx = np.where(labels == c)[0]
    kw = [t for t, _ in topic_model.get_topic(c)[:10]]
    cards.append(cluster_card(f"C{c:02d}", idx, "cluster", kw))
cards.sort(key=lambda d: -d["records"])
print(f"{len(cards)} clusters summarised")

In [ ]:
# ---- Outliers (-1): group into mini-themes, flag data-quality issues ----
from sklearn.cluster import AgglomerativeClustering
out_idx = np.where(labels == -1)[0]
og = AgglomerativeClustering(n_clusters=None, metric="cosine", linkage="average",
                             distance_threshold=CONFIG["outlier_group_distance"]).fit(E[out_idx]).labels_
unique_notes["outlier_group"] = pd.NA
unique_notes.loc[out_idx, "outlier_group"] = og

ctf = CountVectorizer(ngram_range=(1, 2), stop_words=KEYWORD_STOP, min_df=1)
outlier_cards, singles = [], []
for g in pd.Series(og).value_counts().index:
    idx = out_idx[og == g]
    if len(idx) >= 3 or w[idx].sum() >= 5:
        try:
            X = ctf.fit_transform(unique_notes.loc[idx, "note_clean"])
            kw = list(pd.Series(np.asarray(X.multiply(w[idx][:, None]).sum(0)).ravel(), index=ctf.get_feature_names_out()).nlargest(8).index)
        except ValueError:      # group made only of stop words, e.g. "Other"
            kw = []
        outlier_cards.append(cluster_card(f"O{len(outlier_cards):02d}", idx, "outlier_group", kw))
    else:
        singles.extend(idx.tolist())
outlier_cards.sort(key=lambda d: -d["records"])

sing = unique_notes.loc[singles]
dq = pd.DataFrame({
    "issue": ["pasted email / long text (>50 words)", "contains a URL only or mostly", "test / keyboard junk",
              "ungrouped single outliers"],
    "unique_notes": [int((sing["note_clean"].str.split().str.len() > 50).sum()),
                     int(sing["note_clean"].str.contains(r"https?://", case=False).sum()),
                     int(sing["note_clean"].str.fullmatch(r"(?i)(test( only)?|[a-z]{1,8})").sum()),
                     len(sing)]})
print(f"Outliers: {len(out_idx)} notes / {w[out_idx].sum()} tickets -> {len(outlier_cards)} mini-themes + {len(singles)} ungrouped notes")
display(dq)
# ungrouped singles go to the LLM as one sampled card so that unusual reasons still get looked at
rng = np.random.default_rng(CONFIG["random_seed"])
single_sample = rng.choice(singles, size=min(60, len(singles)), replace=False) if singles else []
outlier_singles_card = {"cluster_id": "O-SINGLES", "kind": "outlier_singles", "unique_notes": len(singles),
                        "records": int(w[singles].sum()) if singles else 0, "keywords": [],
                        "sample_notes": [fmt(i) for i in single_sample]}

In [ ]:
summary = pd.DataFrame([{k: c[k] for k in ("cluster_id", "kind", "unique_notes", "records", "mean_within_similarity", "mean_membership_probability")}
                        | {"keywords": ", ".join(c["keywords"][:6]), "top_note": c["representative_notes"][0][:90]}
                        for c in cards + outlier_cards])
display(summary)
(ART / "05_cluster_cards.json").write_text(json.dumps({"clusters": cards, "outlier_groups": outlier_cards,
    "outlier_singles": outlier_singles_card}, indent=1, ensure_ascii=False), encoding="utf-8")
unique_notes.to_pickle(ART / "05_unique_notes.pkl")
RUN["status"]["phase5"] = "done"

## Phase 6 · LLM taxonomy discovery → consolidation → human review

**6a.** Every cluster card (plus outlier mini-themes and a sample of ungrouped outliers) goes to the LLM with the discovery prompt. The LLM returns one JSON object per cluster, which is validated against `DISCOVERY_SCHEMA`.
**6b.** A second pass consolidates the cluster verdicts into one proposed taxonomy, with definitions, inclusion/exclusion rules, examples and subcategories.
**6c.** The proposed taxonomy is exported to `taxonomy/taxonomy_review_v1.xlsx` for a human to edit. Classification (Phase 7) reads **only** the approved JSON built from that file, never the LLM proposal directly.

In [ ]:
BASELINE = {
    "RESIGNATION": "An individual has left their role or organization, resigned, retired, or is no longer responsible for the service or account.",
    "SERVICE_DISSATISFACTION": "The customer is unhappy with the service, product, support, quality, performance, experience, or perceived value.",
    "BUSINESS_REASON": "A business decision, organizational change, financial circumstance, operational requirement, or other commercial consideration has led to disengagement.",
    "FOR_RECORD_PURPOSE_ONLY": "The ticket exists for administrative, documentation, tracking, or procedural purposes rather than to communicate an actual reason for disengagement.",
    "DELETED": "The note explicitly indicates that a ticket, account, service, or record was deleted or removed, without providing a clearer underlying reason.",
    "BLANK": "The note is empty, null, whitespace-only, or contains no meaningful information.",
}

TAXONOMY_DISCOVERY_PROMPT = '''# Role
You are a customer churn analytics and taxonomy discovery specialist.

# Context
Cloudstaff is an offshore staffing provider. Each ticket is a "disengagement": one Cloudstaffer's engagement with a client ends
(staff resigns, client releases the staff, client leaves Cloudstaff, or an administrative re-papering). The note is free text typed by
account managers, sometimes a pasted client email, sometimes a form value ("Voluntary", "Involuntary", "For Record Purpose Only").

# Objective
Identify, group and categorise the underlying reasons for disengagement. Use the baseline categories as a starting point, but do NOT
assume they cover every reason; propose new categories where the data supports them.

# Baseline categories
{baseline}

# Rules
1. Test each cluster against the baseline first. 2. Do not force a cluster into a baseline category it does not reasonably fit.
3. Identify recurring reasons outside the baseline; propose new categories only when several notes show a distinct, recurring reason.
4. Do not create a category per wording variant or isolated example; consolidate synonyms.
5. Distinguish the immediate reason from the underlying business reason when the notes support it.
6. Preserve distinctions such as resignation, service dissatisfaction, pricing, budget constraints, business closure, competitor
   displacement, low adoption and restructuring - only where the notes show them.
7. Do not assume dissatisfaction because someone stopped responding or stopped using the service. Do not infer unsupported information.
8. Notes with several reasons: identify the dominant reason and contributing reasons.
9. A cluster with materially different reasons is MIXED and should be SPLIT.
10. Use UNKNOWN when a meaningful reason exists but cannot be mapped confidently. Keep BLANK and FOR_RECORD_PURPOSE_ONLY distinct.
11. Separate administrative ticket activity from real disengagement reasons.

# Input
Per cluster: id, unique-note count, record (ticket) count, keywords, representative / closest / furthest notes. "[n]" before a note =
number of tickets with that exact note. Cluster ids starting with "O" are outlier mini-themes; "O-SINGLES" is a sample of ungrouped outliers.

# Output
A JSON array, one object per input cluster, each matching:
{{"cluster_id": str, "cluster_size": int, "dominant_theme": str,
 "baseline_category_match": "RESIGNATION|SERVICE_DISSATISFACTION|BUSINESS_REASON|FOR_RECORD_PURPOSE_ONLY|DELETED|BLANK|NONE",
 "proposed_category": str, "category_type": "EXISTING|NEW|UNKNOWN", "category_definition": str,
 "proposed_subcategories": [str], "supporting_examples": [str], "contradictory_examples": [str],
 "cluster_type": "PURE|MIXED|UNCERTAIN", "confidence": 0-1, "recommended_action": "KEEP|MERGE|SPLIT|REVIEW"}}
'''.format(baseline="\n".join(f"- {k}: {v}" for k, v in BASELINE.items()))

DISCOVERY_ITEM = {
    "type": "object",
    "required": ["cluster_id", "cluster_size", "dominant_theme", "baseline_category_match", "proposed_category", "category_type",
                 "category_definition", "proposed_subcategories", "supporting_examples", "contradictory_examples",
                 "cluster_type", "confidence", "recommended_action"],
    "properties": {
        "cluster_id": {"type": "string"}, "cluster_size": {"type": "integer", "minimum": 0},
        "dominant_theme": {"type": "string", "minLength": 3},
        "baseline_category_match": {"enum": list(BASELINE) + ["NONE"]},
        "proposed_category": {"type": "string", "pattern": "^[A-Z][A-Z_]+$"},
        "category_type": {"enum": ["EXISTING", "NEW", "UNKNOWN"]},
        "category_definition": {"type": "string", "minLength": 10},
        "proposed_subcategories": {"type": "array", "items": {"type": "string"}},
        "supporting_examples": {"type": "array", "items": {"type": "string"}, "minItems": 1},
        "contradictory_examples": {"type": "array", "items": {"type": "string"}},
        "cluster_type": {"enum": ["PURE", "MIXED", "UNCERTAIN"]},
        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
        "recommended_action": {"enum": ["KEEP", "MERGE", "SPLIT", "REVIEW"]},
        "split_into": {"type": "array", "items": {"type": "string"}},
    }}
DISCOVERY_SCHEMA = {"type": "array", "items": DISCOVERY_ITEM, "minItems": 1}

all_cards = cards + outlier_cards + [outlier_singles_card]
discovery_payload = [{k: v for k, v in c.items() if k != "kind"} | {"kind": c["kind"]} for c in all_cards]
try:
    discovery = llm_exchange("06a_taxonomy_discovery", TAXONOMY_DISCOVERY_PROMPT, discovery_payload, DISCOVERY_SCHEMA)
except LLMResponsePending as e:
    discovery = None; print(e)

In [ ]:
if discovery is not None:
    disc = pd.DataFrame(discovery)
    sizes = {c["cluster_id"]: c["records"] for c in all_cards}
    missing = set(sizes) - set(disc["cluster_id"]); extra = set(disc["cluster_id"]) - set(sizes)
    bad_size = disc[disc["cluster_size"] != disc["cluster_id"].map(sizes)]
    if missing or extra or len(bad_size):
        raise ValueError(f"Discovery response incomplete: missing={missing} extra={extra} size_mismatch={bad_size['cluster_id'].tolist()}")
    print(f"{len(disc)} cluster verdicts validated")
    display(disc.groupby(["proposed_category"]).agg(clusters=("cluster_id", "size"), records=("cluster_size", "sum"))
            .sort_values("records", ascending=False))
    display(pd.crosstab(disc["cluster_type"], disc["recommended_action"]))
    print("Clusters flagged MIXED / SPLIT:")
    display(disc.loc[disc["cluster_type"].ne("PURE") | disc["recommended_action"].eq("SPLIT"),
                     ["cluster_id", "cluster_size", "dominant_theme", "proposed_category", "cluster_type", "recommended_action", "split_into"]])

In [ ]:
TAXONOMY_CONSOLIDATION_PROMPT = '''You receive the cluster-level taxonomy verdicts produced earlier (JSON array) for Cloudstaff disengagement notes.
Consolidate them into ONE practical taxonomy:
- merge clusters/categories that are the same underlying reason; split categories that are too broad;
- keep categories separate when they reflect different business realities (who initiated, controllable vs market, admin vs churn);
- use parent categories with subcategories rather than many narrow labels;
- always include the six baseline categories (they may be empty or redefined only with justification), plus UNKNOWN;
- give each category: id, name, definition, inclusion rules, exclusion rules, example notes, subcategories, origin (BASELINE|NEW|SYSTEM),
  supporting cluster ids and an estimated ticket count;
- list open questions a human reviewer should decide.
Return JSON: {"categories": [...], "open_questions": [str]}'''

CATEGORY_ITEM = {
    "type": "object",
    "required": ["category_id", "category_name", "definition", "inclusion_rules", "exclusion_rules", "example_notes",
                 "subcategories", "origin", "supporting_cluster_ids", "estimated_records"],
    "properties": {
        "category_id": {"type": "string", "pattern": "^[A-Z][A-Z_]+$"},
        "category_name": {"type": "string", "minLength": 3},
        "definition": {"type": "string", "minLength": 15},
        "inclusion_rules": {"type": "array", "items": {"type": "string"}, "minItems": 1},
        "exclusion_rules": {"type": "array", "items": {"type": "string"}},
        "example_notes": {"type": "array", "items": {"type": "string"}},
        "subcategories": {"type": "array", "items": {"type": "string"}},
        "origin": {"enum": ["BASELINE", "NEW", "SYSTEM"]},
        "supporting_cluster_ids": {"type": "array", "items": {"type": "string"}},
        "estimated_records": {"type": "integer", "minimum": 0},
        "reviewer_notes": {"type": "string"},
    }}
CONSOLIDATION_SCHEMA = {"type": "object", "required": ["categories", "open_questions"],
                        "properties": {"categories": {"type": "array", "items": CATEGORY_ITEM, "minItems": 7},
                                       "open_questions": {"type": "array", "items": {"type": "string"}}}}
proposed = None
if discovery is not None:
    try:
        proposed = llm_exchange("06b_taxonomy_consolidation", TAXONOMY_CONSOLIDATION_PROMPT, discovery, CONSOLIDATION_SCHEMA)
    except LLMResponsePending as e:
        print(e)
if proposed is not None:
    ids = [c["category_id"] for c in proposed["categories"]]
    assert len(ids) == len(set(ids)), "duplicate category ids"
    assert set(BASELINE) | {"UNKNOWN"} <= set(ids), "baseline categories and UNKNOWN must be present"
    unmapped = set(disc["cluster_id"]) - {c for cat in proposed["categories"] for c in cat["supporting_cluster_ids"]}
    if unmapped: warn(f"Clusters not referenced by any proposed category: {sorted(unmapped)}")
    prop = pd.DataFrame(proposed["categories"])
    display(prop[["category_id", "category_name", "origin", "estimated_records", "subcategories"]]
            .assign(subcategories=lambda x: x["subcategories"].str.join("; ")))
    print("Open questions for the reviewer:")
    for q in proposed["open_questions"]: print(" -", q)

In [ ]:
# ---- 6c. Export for human review -------------------------------------------
# The review workbook is the human's working copy and can be reopened at any time (see "Revisiting the taxonomy").
# It is only rewritten when missing or on an older layout; then every reviewer column is carried over and the old file backed up.
REVIEW_XLSX = TAX / "taxonomy_review_v1.xlsx"
REVIEW_LAYOUT = 3           # 2 = redirect_to on Cluster mapping; 3 = revisit / revisit_note columns on both sheets
REVIEW_HELP = [
    ("Taxonomy: approve", "Y = keep this category, N = remove it. Every row needs Y or N before approval."),
    ("Taxonomy: category_id / category_name", "Rename freely. Use UPPER_SNAKE_CASE for the id. Redirects that point at the old id must be updated."),
    ("Taxonomy: merge_into", "To merge, put the target category_id here and set approve = N. Rules, examples and clusters carry over."),
    ("Taxonomy: split / add", "Add new rows (approve = Y, definition, at least one inclusion rule). Set approve = N on a row you split up."),
    ("Taxonomy: definition / rules / examples / subcategories", "Edit the text. Lists are separated by ' | '."),
    ("Taxonomy: reviewer_notes", "Anything the classifier should know. It's passed into the classification prompt."),
    ("Cluster mapping: redirect_to", "Pick a category_id from the dropdown to send this cluster to a different category than the LLM proposed. "
                                     "Leave blank to accept proposed_category. Must be an approved category, or one merged into an approved category."),
    ("Cluster mapping: reviewer_notes", "Why you redirected, or a note on the cluster."),
    ("Cluster mapping: what a redirect changes", "Clusters are a discovery aid: a redirect moves the cluster's example notes and supporting-cluster link to the new "
                                                 "category. Each ticket is still classified on its own note."),
    ("Both sheets: revisit / revisit_note", "Set revisit = Y on anything you approved provisionally and want to come back to. It does NOT block approval; "
                                            "flagged items are listed in the output workbook ('Revisit list') until you clear the flag."),
    ("Revisiting later", "Edit this file at any time and re-run the notebook. A changed taxonomy gets a new version, is saved to taxonomy/history/, "
                         "the change log shows what moved, and only the classifications for the new version are recomputed. Earlier versions stay cached."),
]
LIST_COLS = ["inclusion_rules", "exclusion_rules", "example_notes", "subcategories", "supporting_cluster_ids"]
TAX_EDIT_COLS = ["approve", "merge_into", "reviewer_notes", "revisit", "revisit_note"]
MAP_EDIT_COLS = ["redirect_to", "reviewer_notes", "revisit", "revisit_note"]
MAP_COLS = ["cluster_id", "cluster_size", "dominant_theme", "top_notes", "proposed_category", "redirect_to", "reviewer_notes",
            "revisit", "revisit_note", "cluster_type", "recommended_action", "split_into", "confidence"]

def proposed_taxonomy_frame():
    out = prop.copy()
    for c in LIST_COLS: out[c] = out[c].map(lambda v: " | ".join(v))
    out.insert(0, "approve", ""); out.insert(1, "merge_into", "")
    for c in ("reviewer_notes", "revisit", "revisit_note"):
        if c not in out: out[c] = ""
    return out

def proposed_mapping_frame():
    top = {c["cluster_id"]: "\n".join(n[:160] for n in (c.get("representative_notes") or c.get("sample_notes", []))[:3]) for c in all_cards}
    m = disc.copy()
    m["top_notes"] = m["cluster_id"].map(top)
    m["split_into"] = m.get("split_into", pd.Series([None] * len(m))).map(lambda v: ", ".join(v) if isinstance(v, list) else "")
    for c in MAP_EDIT_COLS: m[c] = ""
    return m[MAP_COLS]

def write_review_workbook(tax, cmap, path):
    with pd.ExcelWriter(path, engine="xlsxwriter") as xw:
        tax.to_excel(xw, sheet_name="Taxonomy", index=False)
        cmap.to_excel(xw, sheet_name="Cluster mapping", index=False)
        pd.DataFrame(REVIEW_HELP, columns=["column / action", "how"]).to_excel(xw, sheet_name="How to review", index=False)
        pd.DataFrame({"open_question": proposed["open_questions"]}).to_excel(xw, sheet_name="Open questions", index=False)
        pd.DataFrame({"layout": [REVIEW_LAYOUT]}).to_excel(xw, sheet_name="_meta", index=False)
        wb = xw.book; wrap = wb.add_format({"text_wrap": True, "valign": "top"})
        edit = wb.add_format({"text_wrap": True, "valign": "top", "bg_color": "#FFF7D6"})   # columns the reviewer fills
        id_col = tax.columns.get_loc("category_id")
        id_range = f"=Taxonomy!${chr(65 + id_col)}$2:${chr(65 + id_col)}$200"     # live list: new/renamed categories appear in dropdowns
        ws = xw.sheets["Taxonomy"]; ws.freeze_panes(1, 3); ws.autofilter(0, 0, len(tax), len(tax.columns) - 1)
        widths = {"approve": 9, "merge_into": 14, "category_id": 28, "category_name": 30, "definition": 60, "origin": 10,
                  "estimated_records": 10, "revisit": 8, "revisit_note": 35}
        for i, col in enumerate(tax.columns):
            ws.set_column(i, i, widths.get(col, 45), edit if col in TAX_EDIT_COLS else wrap)
        ws.data_validation(1, tax.columns.get_loc("approve"), 199, tax.columns.get_loc("approve"), {"validate": "list", "source": ["Y", "N"]})
        ws.data_validation(1, tax.columns.get_loc("merge_into"), 199, tax.columns.get_loc("merge_into"), {"validate": "list", "source": id_range, "ignore_blank": True})
        ws.data_validation(1, tax.columns.get_loc("revisit"), 199, tax.columns.get_loc("revisit"), {"validate": "list", "source": ["Y"], "ignore_blank": True})
        ws = xw.sheets["Cluster mapping"]; ws.freeze_panes(1, 1); ws.autofilter(0, 0, len(cmap), len(cmap.columns) - 1)
        mw = {"cluster_id": 10, "cluster_size": 9, "dominant_theme": 45, "top_notes": 70, "proposed_category": 30, "redirect_to": 30,
              "reviewer_notes": 35, "revisit": 8, "revisit_note": 35, "cluster_type": 11, "recommended_action": 11, "split_into": 40, "confidence": 9}
        for i, col in enumerate(cmap.columns):
            ws.set_column(i, i, mw.get(col, 20), edit if col in MAP_EDIT_COLS else wrap)
        rc = cmap.columns.get_loc("redirect_to")
        ws.data_validation(1, rc, len(cmap), rc, {"validate": "list", "source": id_range, "ignore_blank": True,
                                                  "input_title": "Redirect cluster", "input_message": "Pick a category_id, or leave blank to keep proposed_category"})
        vc = cmap.columns.get_loc("revisit")
        ws.data_validation(1, vc, len(cmap), vc, {"validate": "list", "source": ["Y"], "ignore_blank": True})
        xw.sheets["How to review"].set_column(0, 0, 42); xw.sheets["How to review"].set_column(1, 1, 110, wrap)
        xw.sheets["Open questions"].set_column(0, 0, 140, wrap)
        xw.sheets["_meta"].hide()

def review_layout(path):
    try:
        return int(pd.read_excel(path, "_meta")["layout"].iloc[0])
    except Exception:
        return 1

if proposed is not None:
    if not REVIEW_XLSX.exists():
        write_review_workbook(proposed_taxonomy_frame(), proposed_mapping_frame(), REVIEW_XLSX)
        print(f"Proposed taxonomy written for review: {REVIEW_XLSX}")
    elif review_layout(REVIEW_XLSX) < REVIEW_LAYOUT:
        backup = TAX / f"taxonomy_review_v1.backup-{dt.datetime.now():%Y%m%d-%H%M%S}.xlsx"
        backup.write_bytes(REVIEW_XLSX.read_bytes())
        kept_tax = pd.read_excel(REVIEW_XLSX, "Taxonomy", dtype=object)          # reviewer's sheet, kept exactly
        for c in ("revisit", "revisit_note"):
            if c not in kept_tax: kept_tax[c] = ""
        old_map = pd.read_excel(REVIEW_XLSX, "Cluster mapping", dtype=object)
        new_map = proposed_mapping_frame()
        for col in MAP_EDIT_COLS:                                                  # carry over any mapping edits
            if col in old_map:
                new_map[col] = new_map["cluster_id"].map(old_map.set_index("cluster_id")[col]).fillna("")
        write_review_workbook(kept_tax, new_map, REVIEW_XLSX)
        n_app = kept_tax["approve"].astype(str).str.strip().str.upper().isin(["Y", "N"]).sum()
        print(f"Review workbook upgraded to layout {REVIEW_LAYOUT}; your sheets were kept ({n_app} rows approved). Backup: {backup.name}")
    else:
        print(f"{REVIEW_XLSX.name} is up to date - not overwritten (it holds your edits).")

### Revisiting the taxonomy

Approval isn't final. To revisit:
1. Open `taxonomy/taxonomy_review_v1.xlsx`. Change approvals, merges, definitions, rules or redirects, or set **`revisit = Y`** (plus a `revisit_note`) on anything you approved provisionally. A revisit flag doesn't block approval.
2. Re-run the notebook. 6d builds a new approved version if anything changed: the version id is a hash of the content. It also saves a copy to `taxonomy/history/` and prints a change log against the previous approved version.
3. Phase 7 classifies against the current version only. Results for each version are cached separately, so going back to an earlier version doesn't redo work, and stale results are never mixed in.

Open revisit flags are listed on the output workbook's **Revisit list** sheet, and the **Taxonomy history** sheet lists every approved version.

In [ ]:
# ---- 6d. Build the APPROVED taxonomy from the reviewed file ------------------
def _flag(v):
    return str(v).strip().upper() in ("Y", "YES", "TRUE", "1")

def build_approved_taxonomy(path):
    t = pd.read_excel(path, "Taxonomy", dtype=object).fillna("")
    for c in ("revisit", "revisit_note"):
        if c not in t: t[c] = ""
    t["approve"] = t["approve"].astype(str).str.strip().str.upper()
    t["category_id"] = t["category_id"].astype(str).str.strip()
    t["merge_into"] = t["merge_into"].astype(str).str.strip()
    pending = t.loc[~t["approve"].isin(["Y", "N"]), "category_id"].tolist()
    if pending:
        return None, f"{len(pending)} rows still need approve = Y/N: {pending}"
    keep = t[t["approve"] == "Y"].copy()
    approved_ids = set(keep["category_id"])
    final = {r["category_id"]: (r["category_id"] if r["approve"] == "Y" else (r["merge_into"] or None)) for _, r in t.iterrows()}
    bad_merge = {k: v for k, v in final.items() if v and v not in approved_ids}
    if bad_merge:
        return None, f"merge_into must point at an approved category: {bad_merge}"
    for _, r in t[(t["approve"] == "N") & (t["merge_into"] != "")].iterrows():
        tgt = keep["category_id"] == r["merge_into"]
        for c in ["inclusion_rules", "example_notes", "subcategories"]:
            if str(r[c]).strip():
                keep.loc[tgt, c] = keep.loc[tgt, c].astype(str) + " | " + str(r[c])

    m = pd.read_excel(path, "Cluster mapping", dtype=object).fillna("")
    for c in MAP_EDIT_COLS:
        if c not in m: m[c] = ""
    m["redirect_to"] = m["redirect_to"].astype(str).str.strip()
    m["chosen"] = np.where(m["redirect_to"] != "", m["redirect_to"], m["proposed_category"])
    unknown_target = sorted(set(m.loc[m["redirect_to"] != "", "redirect_to"]) - set(final))
    if unknown_target:
        return None, f"redirect_to names categories that are not in the Taxonomy sheet: {unknown_target}"
    m["approved_category"] = m["chosen"].map(lambda c: final.get(c))
    orphan = m[m["approved_category"].isna()]
    if len(orphan):
        return None, ("These clusters point at a removed category - set redirect_to for them: "
                      + ", ".join(f"{r.cluster_id} ({r.chosen})" for r in orphan.itertuples()))
    cluster_mapping = [{"cluster_id": r.cluster_id, "cluster_size": int(r.cluster_size), "proposed_category": r.proposed_category,
                        "redirect_to": r.redirect_to or None, "approved_category": r.approved_category,
                        "reviewer_notes": str(r.reviewer_notes), "revisit": _flag(r.revisit), "revisit_note": str(r.revisit_note)}
                       for r in m.itertuples()]

    cards_by_id = {c["cluster_id"]: c for c in all_cards}
    moved = m[m["redirect_to"] != ""]
    cats = []
    for _, r in keep.iterrows():
        cat = {k: str(r[k]) for k in ["category_id", "category_name", "definition", "origin", "reviewer_notes"]}
        for c in LIST_COLS: cat[c] = [x.strip() for x in str(r[c]).split("|") if x.strip()]
        cid = cat["category_id"]
        cat["supporting_cluster_ids"] = sorted(m.loc[m["approved_category"] == cid, "cluster_id"])
        for mr in moved[moved["approved_category"] == cid].itertuples():
            notes = cards_by_id.get(mr.cluster_id, {}).get("representative_notes", [])[:2]
            cat["example_notes"] += [re.sub(r"^\[\d+\]\s*", "", n)[:200] for n in notes]
        cat["estimated_records"] = int(m.loc[m["approved_category"] == cid, "cluster_size"].sum())   # tickets in clusters mapped here
        cat["revisit"] = _flag(r["revisit"]); cat["revisit_note"] = str(r["revisit_note"])
        cats.append(cat)
    ids = [c["category_id"] for c in cats]
    if len(ids) != len(set(ids)): return None, "duplicate category_id after review"
    for req in ("BLANK", "UNKNOWN"):
        if req not in ids: return None, f"{req} must stay in the approved taxonomy"
    jsonschema.validate({"categories": cats, "open_questions": []}, CONSOLIDATION_SCHEMA)
    # the version covers what changes classification (definitions, rules, examples, mapping), not revisit flags or notes-to-self
    content = {"categories": [{k: v for k, v in c.items() if k not in ("revisit", "revisit_note", "estimated_records")} for c in cats],
               "clusters": [{k: v for k, v in c.items() if k not in ("revisit", "revisit_note", "reviewer_notes")} for c in cluster_mapping]}
    version = f"v1-{sha(json.dumps(content, sort_keys=True, ensure_ascii=False), 8)}"
    return {"taxonomy_version": version, "approved_at": dt.datetime.now().isoformat(timespec="seconds"),
            "source_file": path.name, "categories": cats, "cluster_mapping": cluster_mapping}, None

def taxonomy_changelog(old, new):
    if old is None: return pd.DataFrame([{"change": "first approved version", "item": new["taxonomy_version"], "detail": ""}])
    rows = []
    oc = {c["category_id"]: c for c in old["categories"]}; nc = {c["category_id"]: c for c in new["categories"]}
    rows += [{"change": "category added", "item": k, "detail": nc[k]["category_name"]} for k in nc.keys() - oc.keys()]
    rows += [{"change": "category removed", "item": k, "detail": oc[k]["category_name"]} for k in oc.keys() - nc.keys()]
    for k in nc.keys() & oc.keys():
        for f in ("category_name", "definition", "inclusion_rules", "exclusion_rules", "subcategories", "reviewer_notes"):
            if nc[k].get(f) != oc[k].get(f): rows.append({"change": f"{f} edited", "item": k, "detail": str(nc[k].get(f))[:300]})
        if nc[k].get("revisit") != oc[k].get("revisit"):
            rows.append({"change": "revisit flag " + ("set" if nc[k].get("revisit") else "cleared"), "item": k, "detail": nc[k].get("revisit_note", "")})
    om = {c["cluster_id"]: c for c in old.get("cluster_mapping", [])}
    for c in new.get("cluster_mapping", []):
        o = om.get(c["cluster_id"])
        if o and o["approved_category"] != c["approved_category"]:
            rows.append({"change": "cluster remapped", "item": c["cluster_id"], "detail": f"{o['approved_category']} -> {c['approved_category']}"})
    return pd.DataFrame(rows or [{"change": "no change", "item": new["taxonomy_version"], "detail": ""}])

APPROVED_JSON = TAX / "taxonomy_approved_v1.json"      # always the CURRENT approved version
HIST = TAX / "history"; HIST.mkdir(exist_ok=True)
approved, changelog = None, None
if REVIEW_XLSX.exists():
    previous = json.loads(APPROVED_JSON.read_text(encoding="utf-8")) if APPROVED_JSON.exists() else None
    approved, problem = build_approved_taxonomy(REVIEW_XLSX)
    if problem:
        print("Taxonomy NOT approved yet:", problem)
        if previous:
            print(f"Keeping the previous approved version {previous['taxonomy_version']} until the review file is complete.")
            approved = previous
    else:
        if previous and previous["taxonomy_version"] == approved["taxonomy_version"]:
            approved["approved_at"] = previous["approved_at"]              # same content: keep the original approval time
        changelog = taxonomy_changelog(previous, approved)
        APPROVED_JSON.write_text(json.dumps(approved, indent=1, ensure_ascii=False), encoding="utf-8")
        hist_file = HIST / f"taxonomy_approved_{approved['taxonomy_version']}.json"
        if not hist_file.exists():
            hist_file.write_text(json.dumps(approved, indent=1, ensure_ascii=False), encoding="utf-8")
        red = sum(1 for c in approved["cluster_mapping"] if c["redirect_to"])
        flags = sum(c["revisit"] for c in approved["categories"]) + sum(c["revisit"] for c in approved["cluster_mapping"])
        print(f"Approved taxonomy {approved['taxonomy_version']}: {len(approved['categories'])} categories, "
              f"{red} clusters redirected, {flags} items flagged to revisit")
        print("Change log vs previous approved version:"); display(changelog)
RUN["status"]["phase6"] = f"approved {approved['taxonomy_version']}" if approved else "awaiting human review"

## Phase 7 · Classify every ticket against the approved taxonomy

Each **unique cleaned note** is classified individually by the LLM, and the result is copied to every ticket with that exact note. Identical text always gets an identical answer, so classifying it once per ticket would only add cost.
* **Batches** are grouped by size (`classification_batch_chars`). Each batch is named by a hash of its note keys, so a rerun after a failure only creates new batches for the notes that failed.
* **Cache:** `llm/classification/cache.jsonl`. The key is a hash of (cleaned note, taxonomy version, model, prompt version), so changing the taxonomy or the prompt never reuses stale answers.
* **Validation:** every item is checked against `CLS_ITEM_SCHEMA` and the approved category ids. Invalid or missing items are re-requested and never silently accepted.
* **Review rules** are enforced in code as well as asked of the LLM: confidence below `confidence_threshold`, UNKNOWN type with a real category, and others.
* **BLANK** notes (Phase 2) don't go to the LLM; they get `BLANK` by rule.

In [ ]:
CONFIG.setdefault("classification_batch_chars", 40000)
if approved is None:
    raise RuntimeError("No approved taxonomy - finish Phase 6 review first.")
TAXV = approved["taxonomy_version"]; PV = CONFIG["classification_prompt_version"]; MODEL = CONFIG["llm_model"]
CAT = {c["category_id"]: c for c in approved["categories"]}
CAT_IDS = sorted(CAT)

CLASSIFICATION_PROMPT = '''# Task
Classify Cloudstaff disengagement notes against the APPROVED taxonomy below. Each note explains why one Cloudstaffer's engagement
with a client ended (or is an administrative record). Classify each note on its own text only.

# Rules
- primary_reason: the ONE category that best states the main reason. Use exactly one approved category_id.
- primary_subcategory: one of that category's listed subcategories if the note supports it, else null.
- secondary_reasons: other approved category_ids ONLY if the note explicitly states an additional contributing reason. Never repeat
  the primary. Do not add semantically related categories that are not stated.
- classification_type: EXPLICIT (note directly states the reason), INFERRED (reasonably implied, not stated), UNKNOWN (text exists but
  no reliable reason -> primary_reason must be UNKNOWN, or BLANK if the text is meaningless).
- BLANK = empty / meaningless text (e.g. 'test', random characters). UNKNOWN = meaningful text with no assignable reason. Keep distinct.
- Never infer a reason from the mere fact that the staff or client disengaged. An unknown reason is better than an invented one.
- Follow each category's inclusion and exclusion rules and the reviewer notes.
- evidence: the shortest exact snippet of the note that supports the primary reason ('' if none).
- confidence: 0-1. review_required = true when confidence < {thr}, several primary categories are plausible, the note is ambiguous,
  has several unrelated reasons, does not fit the taxonomy, or EXPLICIT vs INFERRED is unclear. review_reason: short text, else null.

# Approved taxonomy ({taxv})
{taxonomy}

# Output
A JSON array with one object per input note, in any order:
{{"note_key": str, "primary_reason": category_id, "primary_subcategory": str|null, "secondary_reasons": [category_id],
  "classification_type": "EXPLICIT|INFERRED|UNKNOWN", "evidence": str, "confidence": number, "review_required": bool, "review_reason": str|null}}
'''.format(thr=CONFIG["confidence_threshold"], taxv=TAXV, taxonomy=json.dumps(
    [{k: c[k] for k in ("category_id", "category_name", "definition", "inclusion_rules", "exclusion_rules", "subcategories", "reviewer_notes")}
     | {"example_notes": c["example_notes"][:6]} for c in approved["categories"]], indent=1, ensure_ascii=False))

CLS_ITEM_SCHEMA = {
    "type": "object",
    "required": ["note_key", "primary_reason", "primary_subcategory", "secondary_reasons", "classification_type", "evidence",
                 "confidence", "review_required", "review_reason"],
    "properties": {
        "note_key": {"type": "string"}, "primary_reason": {"enum": CAT_IDS},
        "primary_subcategory": {"type": ["string", "null"]},
        "secondary_reasons": {"type": "array", "items": {"enum": CAT_IDS}, "uniqueItems": True},
        "classification_type": {"enum": ["EXPLICIT", "INFERRED", "UNKNOWN"]},
        "evidence": {"type": "string"}, "confidence": {"type": "number", "minimum": 0, "maximum": 1},
        "review_required": {"type": "boolean"}, "review_reason": {"type": ["string", "null"]},
    }}
print(f"Taxonomy {TAXV}: {len(CAT_IDS)} categories; prompt {PV}; prompt length {len(CLASSIFICATION_PROMPT):,} chars")

In [ ]:
CLS_DIR = LLM_DIR / "classification"; CACHE_FILE = CLS_DIR / "cache.jsonl"
def cache_key(note_clean):
    return sha(f"{note_clean}\x1f{TAXV}\x1f{MODEL}\x1f{PV}", 24)

def load_cache():
    out = {}
    if CACHE_FILE.exists():
        for line in CACHE_FILE.read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line); out[rec["key"]] = rec
    return out

def check_item(item, valid_keys):
    # returns (cleaned item, problem) - problem is None when the item is usable
    try:
        jsonschema.validate(item, CLS_ITEM_SCHEMA)
    except jsonschema.ValidationError as e:
        return None, f"schema: {e.message[:120]}"
    if item["note_key"] not in valid_keys:
        return None, "unknown note_key"
    it = dict(item)
    it["secondary_reasons"] = [s for s in it["secondary_reasons"] if s != it["primary_reason"]]
    subs = CAT[it["primary_reason"]]["subcategories"]
    if it["primary_subcategory"] is not None and it["primary_subcategory"] not in subs:
        it["primary_subcategory"] = None
    return it, None

cache = load_cache()
todo = unique_notes[[cache_key(n) not in cache for n in unique_notes["note_clean"]]].copy()
batches, cur, size = [], [], 0
for r in todo.sort_values("note_key").itertuples():                       # deterministic grouping
    L = len(r.note_clean)
    if cur and size + L > CONFIG["classification_batch_chars"]:
        batches.append(cur); cur, size = [], 0
    cur.append(r); size += L
if cur: batches.append(cur)

pending, got, failed = [], 0, []
for b in batches:
    keys = [r.note_key for r in b]
    bid = f"{TAXV}_{PV}_" + sha("|".join(keys), 10)
    payload = [{"note_key": r.note_key, "note_original": str(r.note_original), "note_clean": r.note_clean, "tickets": int(r.records)} for r in b]
    try:
        resp = llm_exchange(bid, CLASSIFICATION_PROMPT, payload, schema={"type": "array"}, subdir="classification")
    except LLMResponsePending:
        pending.append(bid); continue
    by_key = {r.note_key: r for r in b}; seen = set()
    with CACHE_FILE.open("a", encoding="utf-8") as fh:
        for item in resp:
            it, problem = check_item(item, by_key.keys())
            if problem or it["note_key"] in seen:
                failed.append((bid, item.get("note_key"), problem or "duplicate")); continue
            seen.add(it["note_key"])
            rec = {"key": cache_key(by_key[it["note_key"]].note_clean), "note_key": it["note_key"], "taxonomy_version": TAXV,
                   "prompt_version": PV, "model": MODEL, "batch": bid, "result": it}
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n"); got += 1
    for k in by_key.keys() - seen:
        failed.append((bid, k, "missing from response"))
logger.info(f"Classification: {len(batches)} batches, {got} new results, {len(failed)} failed, {len(pending)} pending")
print(f"{len(unique_notes):,} unique notes | cached before this run: {len(unique_notes) - len(todo):,} | "
      f"batches: {len(batches)} | new results: {got} | failed items: {len(failed)} | batches awaiting a response: {len(pending)}")
if failed:
    RUN["errors"].append(f"{len(failed)} classification items failed validation and will be re-requested on the next run")
    display(pd.DataFrame(failed, columns=["batch", "note_key", "problem"]).head(20))
if pending:
    print("Awaiting LLM responses for:", *pending, sep="\n  ")

In [ ]:
cache = load_cache()
res = {}
for n, k in zip(unique_notes["note_clean"], unique_notes["note_key"]):
    rec = cache.get(cache_key(n))
    if rec: res[k] = rec["result"]
complete = len(res) == len(unique_notes)
print(f"Classified unique notes: {len(res):,} / {len(unique_notes):,}" + ("" if complete else "  (incomplete - rerun after the pending batches are answered)"))

def apply_review_rules(it):
    reasons = [it["review_reason"]] if it.get("review_reason") else []
    flag = bool(it["review_required"])
    if it["confidence"] < CONFIG["confidence_threshold"]:
        flag = True; reasons.append(f"confidence below {CONFIG['confidence_threshold']}")
    if it["classification_type"] == "UNKNOWN" and it["primary_reason"] not in ("UNKNOWN", "BLANK"):
        flag = True; reasons.append("type UNKNOWN but a reason category was assigned")
    if it["primary_reason"] == "UNKNOWN" and it["classification_type"] != "UNKNOWN":
        flag = True; reasons.append("UNKNOWN category with a non-UNKNOWN type")
    return flag, "; ".join(dict.fromkeys(reasons)) or None

OUT_COLS = ["primary_reason_id", "primary_reason_name", "primary_subcategory", "secondary_reason_ids", "secondary_reason_names",
            "classification_type", "classification_evidence", "classification_confidence", "review_required", "review_reason", "classification_source"]
name = lambda cid: CAT[cid]["category_name"] if cid in CAT else cid
rows = []
for k, it in res.items():
    flag, why = apply_review_rules(it)
    rows.append({"note_key": k, "primary_reason_id": it["primary_reason"], "primary_reason_name": name(it["primary_reason"]),
                 "primary_subcategory": it["primary_subcategory"],
                 "secondary_reason_ids": ", ".join(it["secondary_reasons"]),
                 "secondary_reason_names": ", ".join(name(s) for s in it["secondary_reasons"]),
                 "classification_type": it["classification_type"], "classification_evidence": it["evidence"],
                 "classification_confidence": float(it["confidence"]), "review_required": flag, "review_reason": why,
                 "classification_source": "llm"})
cls = pd.DataFrame(rows, columns=["note_key"] + OUT_COLS)

df = df.drop(columns=[c for c in OUT_COLS if c in df.columns]).merge(cls, on="note_key", how="left")
blank = df["processing_status"] == "BLANK"
df.loc[blank, OUT_COLS] = ["BLANK", name("BLANK"), None, "", "", "UNKNOWN", "", 1.0, False, None, "rule: blank note"]
MAIN_MAP = CONFIG["main_category_map"]
unmapped = sorted(set(CAT_IDS) - set(MAIN_MAP))
if unmapped:
    warn(f"Categories missing from CONFIG['main_category_map'] (main_category left blank): {unmapped}")
df["main_category"] = df["primary_reason_id"].map(MAIN_MAP)
df["taxonomy_version"] = TAXV; df["classification_prompt_version"] = PV; df["llm_model"] = MODEL
assert df["record_id"].is_unique and len(df) == len(valid) + blank.sum()
print(f"Tickets with a classification: {df['primary_reason_id'].notna().sum():,} / {len(df):,}")
display(df["primary_reason_id"].value_counts().rename_axis("primary_reason_id").reset_index(name="tickets")
        .assign(pct=lambda x: (x["tickets"] / len(df) * 100).round(1)))
df.to_pickle(ART / f"07_classified_{TAXV}.pkl")
RUN["status"]["phase7"] = "complete" if complete else f"incomplete ({len(res)}/{len(unique_notes)} notes)"

## Phase 8 · Human validation sample and metrics

Sampling is done **by distinct note**, because a ticket-level sample would be mostly "Involuntary" repeated. About 500 notes are stratified across every primary category (rare ones included), confidence high and low, multi-reason, UNKNOWN, outliers (-1), large and small clusters, and review-flagged. One ticket represents each note, and `tickets_with_same_note` shows how far a verdict carries.

The sample is written **once per taxonomy version** to `outputs/validation_<version>.xlsx`, and a rerun never overwrites it. Reviewers fill `human_primary_reason`, `human_secondary_reasons`, `correct` and `comments`. Metrics are computed only from reviewed rows. Operational metrics are always shown.

Two scores are reported. **Accuracy (exact match)** needs the predicted primary to equal the reviewer's. **Accuracy (reviewer rules)** also accepts the reviewer-agreed allowances in CONFIG: a performance + attendance note is right when one label is primary and the other secondary (`validation_equivalent_pairs`), and ROLE_NO_LONGER_NEEDED counts as BUSINESS_REASON at large (`validation_rollup`). Per-category precision / recall stay exact-match.

The sample above was used to write the taxonomy rules, so its score flatters the classifier. A **fresh spot-check** (`outputs/spotcheck_<version>.xlsx`, `spotcheck_size` random notes that were never in any validation file) gives the honest estimate. It's reviewed **blind**: the predictions sit in hidden columns at the right, so pick the category before unhiding them.

Both reviewed files are combined into a **gold dataset** (`outputs/gold_labels.xlsx` and `.csv`), rebuilt on every run so later label edits flow in. It holds the reviewer's labels plus the current model's answers in separate `model_*` columns, so the labels stay fixed when the taxonomy changes.

The reviewed labels are the only ground truth here. The form's structured fields aren't used as truth, because they're unverified before April 2024.

In [ ]:
rng = np.random.default_rng(CONFIG["random_seed"])
note_view = (df[df["processing_status"] == "VALID_TEXT"].groupby("note_key")
             .agg(record_id=("record_id", "first"), tickets=("record_id", "size"), cluster_id=("cluster_id", "first"),
                  primary=("primary_reason_id", "first"), secondary=("secondary_reason_ids", "first"),
                  conf=("classification_confidence", "first"), review=("review_required", "first"))
             .dropna(subset=["primary"]).reset_index())
csize = note_view.groupby("cluster_id")["tickets"].transform("sum")
note_view["stratum_flags"] = ""
def take(mask, n, label):
    pool = note_view[mask & ~note_view["note_key"].isin(chosen)]
    if len(pool) == 0: return
    pick = pool.sample(n=min(n, len(pool)), random_state=int(rng.integers(1_000_000_000)))
    for k in pick["note_key"]:
        chosen[k] = chosen.get(k, label)
chosen = {}
if complete:
    N = CONFIG["validation_sample_size"]
    for cat_id, g in note_view.groupby("primary"):                       # every category, rare ones fully covered up to 12
        take(note_view["primary"].eq(cat_id) & note_view["conf"].ge(0.85), 6, f"category:{cat_id} high-conf")
        take(note_view["primary"].eq(cat_id), 6, f"category:{cat_id}")
    take(note_view["conf"] < CONFIG["confidence_threshold"], 50, "low confidence")
    take(note_view["secondary"].fillna("").astype(str).str.len() > 0, 30, "multi-reason")
    take(note_view["primary"].eq("UNKNOWN"), 20, "unknown")
    take(note_view["cluster_id"].eq(-1), 40, "outlier (-1)")
    take(note_view["review"].astype(bool), 30, "review flagged")
    take(csize >= csize.quantile(0.75), 25, "large cluster")
    take(csize <= csize.quantile(0.25), 25, "small cluster")
    take(note_view["conf"] >= 0.9, 20, "high confidence")
    if len(chosen) < N:                                                 # fill, weighted by ticket volume
        rest = note_view[~note_view["note_key"].isin(chosen)]
        pick = rest.sample(n=min(N - len(chosen), len(rest)), weights=np.sqrt(rest["tickets"]), random_state=CONFIG["random_seed"])
        for k in pick["note_key"]: chosen[k] = "volume-weighted fill"
    print(f"Validation sample: {len(chosen)} distinct notes")

In [ ]:
VAL_XLSX = OUT / f"validation_{TAXV}.xlsx"
VAL_COLS = ["validation_id", "sample_stratum", "record_id", "tickets_with_same_note", "cluster_id", "note_original", "note_clean",
            "predicted_primary_reason", "predicted_subcategory", "predicted_secondary_reasons", "predicted_type", "predicted_confidence",
            "predicted_evidence", "review_required", "human_primary_reason", "human_secondary_reasons", "correct", "reviewer_comments"]
HUMAN_COLS = ["human_primary_reason", "human_secondary_reasons", "correct", "reviewer_comments"]
def previous_reviewed_sample():
    # the most recent validation file of an EARLIER taxonomy version that holds reviewer labels
    for pv in sorted((p for p in OUT.glob("validation_*.xlsx") if p != VAL_XLSX and not p.name.startswith("~$")),
                     key=lambda p: p.stat().st_mtime, reverse=True):
        d = pd.read_excel(pv, "Validation", dtype=object)
        if d["human_primary_reason"].notna().any():
            return d, pv.name
    return None, None

carried = None
if complete and not VAL_XLSX.exists():
    carried, carried_from = previous_reviewed_sample()
    if carried is not None:
        # Reuse the reviewed notes so a new taxonomy version is scored on the same ground truth.
        # 'correct' judged the OLD prediction, so it is cleared; the human labels carry over as they are.
        carried = carried.assign(record_id=carried["record_id"].astype(str), _order=range(len(carried)))
        s = df.assign(record_id=df["record_id"].astype(str)).merge(
            carried[["record_id", "sample_stratum", "_order"]], on="record_id").merge(note_view[["note_key", "tickets"]], on="note_key")
        s = s.sort_values("_order")
        print(f"Reusing the {len(s)} reviewed notes from {carried_from} (labels carried over, 'correct' cleared).")
    else:
        s = df[df["note_key"].isin(chosen)].drop_duplicates("note_key").copy()
        s["sample_stratum"] = s["note_key"].map(chosen)
        s = s.merge(note_view[["note_key", "tickets"]], on="note_key").sort_values(["sample_stratum", "primary_reason_id"])
    val = pd.DataFrame({
        "validation_id": [f"V{i:04d}" for i in range(1, len(s) + 1)], "sample_stratum": s["sample_stratum"].values,
        "record_id": s["record_id"].values, "tickets_with_same_note": s["tickets"].values, "cluster_id": s["cluster_id"].values,
        "note_original": s[TEXT].astype(str).str.replace("_x000D_", "", regex=False).values, "note_clean": s["note_clean"].values,
        "predicted_primary_reason": s["primary_reason_id"].values, "predicted_subcategory": s["primary_subcategory"].values,
        "predicted_secondary_reasons": s["secondary_reason_ids"].values, "predicted_type": s["classification_type"].values,
        "predicted_confidence": s["classification_confidence"].values, "predicted_evidence": s["classification_evidence"].values,
        "review_required": s["review_required"].values,
        "human_primary_reason": "", "human_secondary_reasons": "", "correct": "", "reviewer_comments": ""})
    if carried is not None:
        lab = carried.set_index("record_id")
        for col in ("human_primary_reason", "human_secondary_reasons", "reviewer_comments"):
            val[col] = val["record_id"].astype(str).map(lab[col]).where(lambda x: x.notna(), "")
    with pd.ExcelWriter(VAL_XLSX, engine="xlsxwriter") as xw:
        val.to_excel(xw, sheet_name="Validation", index=False)
        pd.DataFrame({"category_id": CAT_IDS, "category_name": [name(c) for c in CAT_IDS]}).to_excel(xw, sheet_name="Categories", index=False)
        wb = xw.book; ws = xw.sheets["Validation"]
        wrap = wb.add_format({"text_wrap": True, "valign": "top"}); edit = wb.add_format({"text_wrap": True, "valign": "top", "bg_color": "#FFF7D6"})
        for i, c in enumerate(VAL_COLS):
            w = 60 if c in ("note_original",) else 35 if c in ("predicted_evidence", "reviewer_comments", "note_clean") else 22
            ws.set_column(i, i, w, edit if c.startswith("human_") or c in ("correct", "reviewer_comments") else wrap)
        ws.set_column(VAL_COLS.index("note_clean"), VAL_COLS.index("note_clean"), 35, wrap, {"hidden": True})
        ws.freeze_panes(1, 3); ws.autofilter(0, 0, len(val), len(VAL_COLS) - 1)
        hp = VAL_COLS.index("human_primary_reason"); cc = VAL_COLS.index("correct")
        ws.data_validation(1, hp, len(val), hp, {"validate": "list", "source": f"=Categories!$A$2:$A${len(CAT_IDS) + 1}", "ignore_blank": True})
        ws.data_validation(1, cc, len(val), cc, {"validate": "list", "source": ["Y", "N"], "ignore_blank": True})
        xw.sheets["Categories"].set_column(0, 1, 40)
    print(f"Validation sample written: {VAL_XLSX}")
elif VAL_XLSX.exists():
    print(f"{VAL_XLSX.name} exists - kept (it may hold reviewer labels).")
val = pd.read_excel(VAL_XLSX, "Validation", dtype=object) if VAL_XLSX.exists() else pd.DataFrame(columns=VAL_COLS)

In [ ]:
# ---- 8b. Fresh blind spot-check ------------------------------------------------
# Simple random sample of unique notes that were never in ANY validation file, so the rules were not tuned on them.
SPOT_XLSX = OUT / f"spotcheck_{TAXV}.xlsx"
SPOT_COLS = ["spot_id", "record_id", "tickets_with_same_note", "note_original", "human_primary_reason", "human_secondary_reasons",
             "reviewer_comments", "predicted_primary_reason", "predicted_secondary_reasons", "predicted_confidence"]
def keys_in_validation_files():
    ids = set()
    for pv in OUT.glob("validation_*.xlsx"):
        if not pv.name.startswith("~$"):
            ids |= set(pd.read_excel(pv, "Validation", dtype=object)["record_id"].astype(str))
    return set(df.loc[df["record_id"].astype(str).isin(ids), "note_key"])

if complete and CONFIG.get("spotcheck_size") and not SPOT_XLSX.exists():
    pool = note_view[~note_view["note_key"].isin(keys_in_validation_files())]
    pick = pool.sample(n=min(CONFIG["spotcheck_size"], len(pool)), random_state=CONFIG["random_seed"] + 1)
    first = df.drop_duplicates("note_key").set_index("note_key")
    spot = pd.DataFrame({
        "spot_id": [f"S{i:03d}" for i in range(1, len(pick) + 1)], "record_id": pick["record_id"].values,
        "tickets_with_same_note": pick["tickets"].values,
        "note_original": first.loc[pick["note_key"], TEXT].astype(str).str.replace("_x000D_", "", regex=False).values,
        "human_primary_reason": "", "human_secondary_reasons": "", "reviewer_comments": "",
        "predicted_primary_reason": pick["primary"].values, "predicted_secondary_reasons": pick["secondary"].fillna("").values,
        "predicted_confidence": pick["conf"].values})
    with pd.ExcelWriter(SPOT_XLSX, engine="xlsxwriter") as xw:
        spot.to_excel(xw, sheet_name="Spot check", index=False)
        pd.DataFrame([{"category_id": c, "category_name": name(c), "definition": CAT[c]["definition"]} for c in CAT_IDS]) \
          .to_excel(xw, sheet_name="Categories", index=False)
        wb = xw.book; ws = xw.sheets["Spot check"]
        wrap = wb.add_format({"text_wrap": True, "valign": "top"}); edit = wb.add_format({"text_wrap": True, "valign": "top", "bg_color": "#FFF7D6"})
        for i, c in enumerate(SPOT_COLS):
            w = 80 if c == "note_original" else 35 if c.startswith("human_") or c == "reviewer_comments" else 14
            ws.set_column(i, i, w, edit if c.startswith("human_") or c == "reviewer_comments" else wrap,
                          {"hidden": c.startswith("predicted_")})                      # blind review: unhide only after labelling
        hp = SPOT_COLS.index("human_primary_reason")
        ws.data_validation(1, hp, len(spot), hp, {"validate": "list", "source": f"=Categories!$A$2:$A${len(CAT_IDS) + 1}", "ignore_blank": True})
        ws.freeze_panes(1, 4)
        xs = xw.sheets["Categories"]; xs.set_column(0, 1, 32); xs.set_column(2, 2, 100, wrap)
    print(f"Spot-check written: {SPOT_XLSX} ({len(spot)} notes, predictions hidden)")
elif SPOT_XLSX.exists():
    print(f"{SPOT_XLSX.name} exists - kept (it may hold reviewer labels).")
spot = pd.read_excel(SPOT_XLSX, "Spot check", dtype=object) if SPOT_XLSX.exists() else pd.DataFrame(columns=SPOT_COLS)


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.preprocessing import MultiLabelBinarizer

def operational_metrics(d):
    n = len(d); nb = d[d["processing_status"] == "VALID_TEXT"]
    rows = [("Total tickets", n, 100.0),
            ("BLANK", int((d["primary_reason_id"] == "BLANK").sum()), None),
            ("UNKNOWN / unclassified", int((d["primary_reason_id"] == "UNKNOWN").sum()), None),
            ("Flagged for review", int(d["review_required"].fillna(False).astype(bool).sum()), None),
            ("EXPLICIT (non-blank notes)", int((nb["classification_type"] == "EXPLICIT").sum()), None),
            ("INFERRED (non-blank notes)", int((nb["classification_type"] == "INFERRED").sum()), None),
            ("UNKNOWN type (non-blank notes)", int((nb["classification_type"] == "UNKNOWN").sum()), None),
            ("With at least one secondary reason", int((d["secondary_reason_ids"].fillna("") != "").sum()), None)]
    out = pd.DataFrame(rows, columns=["metric", "tickets", "pct"])
    out["pct"] = (out["tickets"] / n * 100).round(1)
    return out

def split_ids(v):
    return sorted({x.strip() for x in str(v).replace(";", ",").split(",") if x.strip() and x.strip().lower() != "nan"})

EQ_PAIRS = [frozenset(x) for x in CONFIG.get("validation_equivalent_pairs", [])]
ROLLUP = CONFIG.get("validation_rollup", {})
def reviewer_rule_match(h, p, sec):
    # exact match, or a reviewer-agreed allowance (see CONFIG)
    if h == p or ROLLUP.get(h, h) == ROLLUP.get(p, p):
        return True
    return frozenset((h, p)) in EQ_PAIRS and h in sec

def rule_scores(rev, prefix=""):
    h, p = rev["human_primary_reason"].tolist(), rev["predicted_primary_reason"].tolist()
    sec = rev["predicted_secondary_reasons"].map(split_ids).tolist()
    exact = np.array([a == b for a, b in zip(h, p)]); rule = np.array([reviewer_rule_match(a, b, s) for a, b, s in zip(h, p, sec)])
    anyl = rule | np.array([a in s for a, s in zip(h, sec)])
    mm = CONFIG["main_category_map"]
    main = np.array([mm.get(a, a) == mm.get(b, b) for a, b in zip(h, p)])
    return [(f"{prefix}Accuracy (exact match)", round(exact.mean(), 3)), (f"{prefix}Accuracy (reviewer rules)", round(rule.mean(), 3)),
            (f"{prefix}Reviewer label anywhere in primary / secondary", round(anyl.mean(), 3)),
            (f"{prefix}Accuracy (main category)", round(main.mean(), 3))], rule

def spot_metrics(s):
    s = s.copy()
    for c in ("human_primary_reason", "predicted_secondary_reasons"):
        s[c] = s[c].fillna("").astype(str).str.strip()
    rev = s[s["human_primary_reason"] != ""]
    if rev.empty:
        return None
    rows, rule = rule_scores(rev, "Fresh spot-check: ")
    w = pd.to_numeric(rev["tickets_with_same_note"], errors="coerce").fillna(1).to_numpy()
    rows = [("Fresh spot-check: reviewed notes", len(rev))] + rows + \
           [("Fresh spot-check: Accuracy (reviewer rules), ticket-weighted", round(float((rule * w).sum() / w.sum()), 3))]
    return pd.DataFrame(rows, columns=["metric", "value"])

def validation_metrics(v):
    v = v.copy()
    for c in ("correct", "human_primary_reason", "human_secondary_reasons", "predicted_secondary_reasons"):
        v[c] = v[c].fillna("").astype(str).str.strip()          # pandas 3 keeps NaN through astype(str), so fill first
    v["correct"] = v["correct"].str.upper()
    v.loc[(v["human_primary_reason"] == "") & (v["correct"] == "Y"), "human_primary_reason"] = v["predicted_primary_reason"]
    rev = v[v["human_primary_reason"] != ""]
    if rev.empty:
        return None
    y, p = rev["human_primary_reason"], rev["predicted_primary_reason"]
    labels = sorted(set(y) | set(p))
    P, R, F, S = precision_recall_fscore_support(y, p, labels=labels, zero_division=0)
    per_cat = pd.DataFrame({"category": labels, "precision": P.round(3), "recall": R.round(3), "f1": F.round(3), "support (human)": S})
    mp, mr, mf, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    rule_rows, _ = rule_scores(rev)
    summary = pd.DataFrame([("Reviewed rows", len(rev))] + rule_rows + [
                            ("Macro precision", round(mp, 3)), ("Macro recall", round(mr, 3)), ("Macro F1", round(mf, 3))], columns=["metric", "value"])
    cm = pd.DataFrame(confusion_matrix(y, p, labels=labels), index=[f"human:{l}" for l in labels], columns=[f"pred:{l}" for l in labels])
    hs = rev["human_secondary_reasons"].map(split_ids); ps = rev["predicted_secondary_reasons"].map(split_ids)
    use = (rev["correct"] == "Y") | (rev["human_secondary_reasons"] != "")
    hs = hs.where(~((rev["correct"] == "Y") & (rev["human_secondary_reasons"] == "")), ps)
    if use.any():
        mlb = MultiLabelBinarizer(classes=CAT_IDS).fit([CAT_IDS])
        sp, sr, sf, _ = precision_recall_fscore_support(mlb.transform(hs[use]), mlb.transform(ps[use]), average="micro", zero_division=0)
        summary = pd.concat([summary, pd.DataFrame([("Secondary reasons: micro precision", round(sp, 3)), ("Secondary reasons: micro recall", round(sr, 3)),
                                                    ("Secondary reasons: micro F1", round(sf, 3))], columns=["metric", "value"])])
    return {"summary": summary, "per_category": per_cat, "confusion": cm}

ops = operational_metrics(df); display(ops)
vm = validation_metrics(val) if len(val) else None
sm = spot_metrics(spot) if len(spot) else None
if vm and sm is not None:
    vm["summary"] = pd.concat([vm["summary"], sm], ignore_index=True)
if vm:
    display(vm["summary"]); display(vm["per_category"])
else:
    print(f"No reviewed validation rows yet - fill {VAL_XLSX.name} and rerun to get accuracy / precision / recall / F1.")
conf_hist = pd.cut(df.loc[df["processing_status"] == "VALID_TEXT", "classification_confidence"],
                   bins=[0, .5, .6, .7, .75, .8, .9, .95, 1.0], include_lowest=True).value_counts().sort_index()
RUN["status"]["phase8"] = (f"validation sample {len(val)} notes; reviewed: {0 if not vm else int(vm['summary'].iloc[0, 1])}; "
                          f"spot-check {len(spot)} notes; reviewed: {0 if sm is None else int(sm.iloc[0, 1])}")

In [ ]:
# ---- 8c. Gold dataset: every human-labelled note in one place --------------------
# Validation sample (stratified; it was used to write the rules) + fresh blind spot-check (simple random).
# gold_* columns are the reviewer's labels and never change with the taxonomy; model_* columns are the CURRENT version's answers.
GOLD_XLSX = OUT / "gold_labels.xlsx"; GOLD_CSV = OUT / "gold_labels.csv"
GOLD_COLS = ["gold_id", "source", "source_id", "sample_stratum", "record_id", "note_key", "tickets_with_same_note", "note_original", "note_clean",
             "gold_primary_reason", "gold_secondary_reasons", "gold_main_category", "reviewer_comments",
             "model_primary_reason", "model_secondary_reasons", "model_main_category", "model_matches_primary", "model_matches_main", "model_taxonomy_version"]

def build_gold(v, s):
    parts = []
    if len(v):
        parts.append(v.assign(source="validation sample", source_id=v["validation_id"]))
    if len(s):
        parts.append(s.assign(source="fresh spot-check (blind)", source_id=s["spot_id"], sample_stratum="simple random"))
    if not parts:
        return pd.DataFrame(columns=GOLD_COLS)
    g = pd.concat(parts, ignore_index=True)
    g["gold_primary_reason"] = g["human_primary_reason"].fillna("").astype(str).str.strip()
    g = g[g["gold_primary_reason"] != ""].copy()
    bad = sorted(set(g["gold_primary_reason"]) - set(CAT_IDS))
    if bad:
        warn(f"Gold labels that are not approved category ids: {bad}")
    tix = df.assign(record_id=df["record_id"].astype(str)).set_index("record_id")
    g["record_id"] = g["record_id"].astype(str)
    g["note_key"] = g["record_id"].map(tix["note_key"])
    dup = g["note_key"].duplicated(keep="first")
    if dup.any():
        warn(f"{int(dup.sum())} notes are labelled in both files; the validation-sample label is kept")
        g = g[~dup]
    g["note_clean"] = g["record_id"].map(tix["note_clean"])
    g["note_original"] = g["record_id"].map(tix[TEXT]).astype(str).str.replace("_x000D_", "", regex=False)
    g["tickets_with_same_note"] = g["note_key"].map(note_view.set_index("note_key")["tickets"])
    g["gold_secondary_reasons"] = g["human_secondary_reasons"].map(lambda x: ", ".join(split_ids(x)))
    g["gold_main_category"] = g["gold_primary_reason"].map(MAIN_MAP)
    g["reviewer_comments"] = g["reviewer_comments"].fillna("").astype(str)
    g["model_primary_reason"] = g["record_id"].map(tix["primary_reason_id"])
    g["model_secondary_reasons"] = g["record_id"].map(tix["secondary_reason_ids"]).fillna("")
    g["model_main_category"] = g["model_primary_reason"].map(MAIN_MAP)
    g["model_matches_primary"] = g["model_primary_reason"] == g["gold_primary_reason"]
    g["model_matches_main"] = g["model_main_category"] == g["gold_main_category"]
    g["model_taxonomy_version"] = TAXV
    g["gold_id"] = [f"G{i:04d}" for i in range(1, len(g) + 1)]
    return g[GOLD_COLS].reset_index(drop=True)

gold = build_gold(val, spot)
if len(gold):
    about = pd.DataFrame([
        ("What", "Every note a human has labelled: the stratified validation sample plus the fresh blind spot-check. One row per distinct note."),
        ("gold_* columns", "The reviewer's labels. They do not change when the taxonomy or the model changes."),
        ("gold_main_category", "gold_primary_reason mapped with CONFIG['main_category_map']."),
        ("model_* columns", f"The model's answers for taxonomy {TAXV}, refreshed every run, for comparison only."),
        ("source", "'validation sample' was stratified (rare categories over-sampled) and used to write the taxonomy rules, so it flatters the model. "
                   "'fresh spot-check (blind)' is a simple random sample of notes the rules were never tuned on: use it for honest accuracy."),
        ("Editing labels", "Edit the source files (validation_<version>.xlsx, spotcheck_<version>.xlsx), not this file; it is rebuilt on every run."),
        ("Built", dt.datetime.now().isoformat(timespec="seconds")),
    ], columns=["item", "detail"])
    with pd.ExcelWriter(GOLD_XLSX, engine="xlsxwriter") as xw:
        gold.to_excel(xw, sheet_name="Gold", index=False); about.to_excel(xw, sheet_name="About", index=False)
        wb = xw.book; wrap = wb.add_format({"text_wrap": True, "valign": "top"}); ws = xw.sheets["Gold"]
        for i, c in enumerate(GOLD_COLS):
            ws.set_column(i, i, 70 if c == "note_original" else 30 if c.startswith(("gold_", "model_")) or c == "reviewer_comments" else 14,
                          wrap if c == "note_original" else None, {"hidden": c == "note_clean"})
        ws.freeze_panes(1, 3); ws.autofilter(0, 0, len(gold), len(GOLD_COLS) - 1)
        xw.sheets["About"].set_column(0, 0, 22); xw.sheets["About"].set_column(1, 1, 120, wrap)
    gold.to_csv(GOLD_CSV, index=False, encoding="utf-8-sig")
    summary_gold = (gold.groupby("source").agg(notes=("gold_id", "size"), tickets=("tickets_with_same_note", "sum"),
                                               model_matches_primary=("model_matches_primary", "mean"), model_matches_main=("model_matches_main", "mean"))
                    .round(3))
    print(f"Gold dataset: {len(gold)} labelled notes -> {GOLD_XLSX.name} / {GOLD_CSV.name}")
    display(summary_gold)
    display(gold["gold_main_category"].value_counts().rename_axis("gold_main_category").reset_index(name="notes"))
RUN["status"]["gold"] = f"{len(gold)} labelled notes"


## Phase 9 · Excel workbook

One workbook, `outputs/ub_reason_classification.xlsx`, rebuilt on every run. It contains:
* **Classified Tickets**: every original column plus the classification columns.
* **Approved Taxonomy** and **Cluster Analysis**.
* **Summary Dashboard**, with native Excel charts.
* **Validation**: a mirror of the validation file, which is where reviewers edit.
* **Cluster vs Category**: how each cluster's tickets were actually classified.
* **Revisit list** and **Taxonomy history**.
* **Profiling** and **Run Metadata**.

In [ ]:
ILLEGAL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
def xl_safe(frame):
    f = frame.copy()
    for c in f.columns:
        if f[c].dtype == object or str(f[c].dtype).startswith("string"):
            f[c] = f[c].map(lambda v: ILLEGAL.sub("", v)[:32000] if isinstance(v, str) else v)
    return f

tickets = df.drop(columns=["note_key"]).rename(columns={"note_kind": "note_kind (EDA)"})
first = list(df.columns[:df.columns.get_loc("note_clean")])
lead = [c for c in first if c != "note_kind"]
new_cols = ["note_clean", "processing_status", "cluster_id"] + OUT_COLS + ["taxonomy_version", "classification_prompt_version", "llm_model"]
new_cols.insert(new_cols.index("primary_reason_name") + 1, "main_category")
tickets = tickets[[c for c in lead if c in tickets.columns] + new_cols + ["note_kind (EDA)"]]

n_by = df["primary_reason_id"].value_counts()
tax_sheet = pd.DataFrame([{
    "category_id": c["category_id"], "category_name": c["category_name"], "main_category": MAIN_MAP.get(c["category_id"], ""),
    "definition": c["definition"],
    "inclusion_rules": " | ".join(c["inclusion_rules"]), "exclusion_rules": " | ".join(c["exclusion_rules"]),
    "examples": " | ".join(c["example_notes"][:6]), "subcategories": " | ".join(c["subcategories"]),
    "baseline_or_new": c["origin"], "supporting_cluster_ids": ", ".join(c["supporting_cluster_ids"]),
    "classified_tickets": int(n_by.get(c["category_id"], 0)), "pct_of_all_tickets": round(n_by.get(c["category_id"], 0) / len(df) * 100, 1),
    "revisit": "Y" if c.get("revisit") else "", "revisit_note": c.get("revisit_note", ""), "reviewer_notes": c.get("reviewer_notes", "")}
    for c in approved["categories"]]).sort_values("classified_tickets", ascending=False)

amap = {c["cluster_id"]: c["approved_category"] for c in approved["cluster_mapping"]}
card = {c["cluster_id"]: c for c in all_cards}
cl_sheet = disc.assign(
    unique_note_count=disc["cluster_id"].map(lambda k: card[k]["unique_notes"]),
    top_keywords=disc["cluster_id"].map(lambda k: ", ".join(card[k].get("keywords", [])[:8])),
    approved_category=disc["cluster_id"].map(amap),
    representative_notes=disc["cluster_id"].map(lambda k: "\n".join((card[k].get("representative_notes") or card[k].get("sample_notes", []))[:5])),
    supporting_examples=disc["supporting_examples"].map(lambda v: " | ".join(v)),
    contradictory_examples=disc["contradictory_examples"].map(lambda v: " | ".join(v)),
)[["cluster_id", "unique_note_count", "cluster_size", "top_keywords", "dominant_theme", "proposed_category", "approved_category",
   "cluster_type", "confidence", "recommended_action", "representative_notes", "supporting_examples", "contradictory_examples"]] \
   .rename(columns={"cluster_size": "original_record_count"})

# cluster vs per-note classification: how far each cluster's tickets agree with the category the cluster was mapped to
cv = df[df["processing_status"] == "VALID_TEXT"].assign(mapped=lambda x: x["cluster_id"].map(
        lambda c: amap.get("O-SINGLES" if c == -1 else f"C{int(c):02d}") if pd.notna(c) else None))
cv.loc[cv["cluster_id"] == -1, "mapped"] = None     # outliers were reviewed as mini-themes, not one cluster
agree = (cv.groupby("cluster_id").apply(lambda g: pd.Series({
            "tickets": len(g), "unique_notes": g["note_key"].nunique(),
            "mapped_category": g["mapped"].iloc[0],
            "pct_tickets_matching_mapped": round((g["primary_reason_id"] == g["mapped"]).mean() * 100, 1) if pd.notna(g["mapped"].iloc[0]) else None,
            "top_classified_category": g["primary_reason_id"].value_counts().index[0],
            "pct_top_category": round(g["primary_reason_id"].value_counts(normalize=True).iloc[0] * 100, 1),
            "categories_present": g["primary_reason_id"].nunique()}), include_groups=False)
         .reset_index())
agree["cluster_id"] = agree["cluster_id"].map(lambda c: "-1 (outliers)" if c == -1 else f"C{int(c):02d}")
agree = agree.merge(disc[["cluster_id", "cluster_type", "dominant_theme"]], on="cluster_id", how="left").sort_values("tickets", ascending=False)
cluster_x_cat = pd.crosstab(cv["cluster_id"].map(lambda c: "-1 (outliers)" if c == -1 else f"C{int(c):02d}"), cv["primary_reason_id"])
print("Cluster vs classification agreement (by ticket):")
display(agree.head(15))
w_ag = agree.dropna(subset=["pct_tickets_matching_mapped"])
print(f"Ticket-weighted share matching the cluster's mapped category: "
      f"{(w_ag['pct_tickets_matching_mapped'] * w_ag['tickets']).sum() / w_ag['tickets'].sum():.1f}% "
      f"| PURE clusters: {(lambda a: (a['pct_tickets_matching_mapped'] * a['tickets']).sum() / a['tickets'].sum())(w_ag[w_ag['cluster_type'] == 'PURE']):.1f}% "
      f"| MIXED/UNCERTAIN: {(lambda a: (a['pct_tickets_matching_mapped'] * a['tickets']).sum() / a['tickets'].sum())(w_ag[w_ag['cluster_type'] != 'PURE']):.1f}%")

revisit = pd.DataFrame([{"type": "category", "id": c["category_id"], "name / category": c["category_name"], "revisit_note": c.get("revisit_note", "")}
                        for c in approved["categories"] if c.get("revisit")] +
                       [{"type": "cluster", "id": c["cluster_id"], "name / category": c["approved_category"], "revisit_note": c.get("revisit_note", "")}
                        for c in approved["cluster_mapping"] if c.get("revisit")], columns=["type", "id", "name / category", "revisit_note"])
history = pd.DataFrame([{"taxonomy_version": (h := json.loads(p.read_text(encoding="utf-8")))["taxonomy_version"],
                         "approved_at": h["approved_at"], "categories": len(h["categories"]),
                         "redirected_clusters": sum(1 for c in h.get("cluster_mapping", []) if c.get("redirect_to")),
                         "current": "Y" if h["taxonomy_version"] == TAXV else ""} for p in sorted(HIST.glob("taxonomy_approved_*.json"))])

prim = df["primary_reason_id"].value_counts().rename_axis("primary_reason_id").reset_index(name="tickets")
prim["primary_reason_name"] = prim["primary_reason_id"].map(name); prim["main_category"] = prim["primary_reason_id"].map(MAIN_MAP); prim["pct"] = (prim["tickets"] / len(df) * 100).round(1)
sec = pd.Series([s for v in df["secondary_reason_ids"].fillna("") for s in split_ids(v)]).value_counts() \
        .rename_axis("secondary_reason_id").reset_index(name="tickets") if (df["secondary_reason_ids"].fillna("") != "").any() \
        else pd.DataFrame(columns=["secondary_reason_id", "tickets"])
ctype = df.loc[df["processing_status"] == "VALID_TEXT", "classification_type"].value_counts().rename_axis("classification_type").reset_index(name="tickets")
confd = conf_hist.rename_axis("confidence_band").reset_index(name="tickets"); confd["confidence_band"] = confd["confidence_band"].astype(str)
kpis = pd.DataFrame([("Total tickets", len(df)), ("Blank notes", int((df["primary_reason_id"] == "BLANK").sum())),
                     ("Classified (not BLANK / UNKNOWN)", int((~df["primary_reason_id"].isin(["BLANK", "UNKNOWN"])).sum())),
                     ("Unknown / unclassified", int((df["primary_reason_id"] == "UNKNOWN").sum())),
                     ("Approved categories", len(CAT_IDS)),
                     ("% flagged for review", round(df["review_required"].fillna(False).astype(bool).mean() * 100, 1)),
                     ("Taxonomy version", TAXV)], columns=["measure", "value"])

meta = pd.DataFrame([
    ("Run started", RUN["started"]), ("Workbook written", dt.datetime.now().isoformat(timespec="seconds")),
    ("Input file", CONFIG["input_path"]), ("Sheet", CONFIG["sheet_name"]), ("Note column", TEXT), ("ID column", CONFIG["id_col"]),
    ("Date filter", f"{CONFIG['date_col']} <= {CONFIG['date_max']}"), ("Total records processed", len(df)),
    ("Unique non-blank notes", len(unique_notes)), ("Embedding model", CONFIG["embedding_model"]),
    ("UMAP params", json.dumps(CONFIG["umap"])), ("HDBSCAN params", json.dumps(CONFIG["hdbscan"] | {"min_cluster_size": SEL})),
    ("Clustering configs compared", ", ".join(map(str, CONFIG["min_cluster_sizes"]))),
    ("LLM backend / model", f"{CONFIG['llm_backend']} / {MODEL}"),
    ("Prompt versions", f"taxonomy {CONFIG['taxonomy_prompt_version']}, classification {PV}"),
    ("Taxonomy version", TAXV), ("Taxonomy approved at", approved["approved_at"]),
    ("Confidence threshold", CONFIG["confidence_threshold"]),
    ("Processing status", json.dumps(RUN["status"])), ("Warnings", " | ".join(RUN["warnings"]) or "none"),
    ("Errors", " | ".join(RUN["errors"]) or "none"),
    ("Note on ground truth", "Only human-reviewed validation rows are treated as truth; structured form fields are unverified before Apr 2024 and are not used."),
], columns=["item", "value"])
print("Sheets prepared")

In [ ]:
WB = OUT / CONFIG["output_workbook"]
def add_table(ws, frame, r0=0, c0=0, name=None, widths=None, wrap_cols=(), fmt=None):
    cols = [{"header": str(c)} for c in frame.columns]
    data = frame.astype(object).where(frame.notna(), None).values.tolist()
    ws.add_table(r0, c0, r0 + max(len(frame), 1), c0 + len(frame.columns) - 1,
                 {"columns": cols, "data": data, "name": name, "style": "Table Style Light 9"})
    for i, c in enumerate(frame.columns):
        w = (widths or {}).get(c, min(max(len(str(c)) + 2, 12), 40))
        ws.set_column(c0 + i, c0 + i, w, fmt.get("wrap") if c in wrap_cols else None)

try:
    with pd.ExcelWriter(WB, engine="xlsxwriter", datetime_format="yyyy-mm-dd", engine_kwargs={"options": {"strings_to_urls": False, "default_date_format": "yyyy-mm-dd"}}) as xw:
        wb = xw.book
        fmt = {"wrap": wb.add_format({"text_wrap": True, "valign": "top"}), "h": wb.add_format({"bold": True, "font_size": 13}),
               "pct": wb.add_format({"num_format": "0.0"})}
        # 1 Classified Tickets
        t = xl_safe(tickets); ws = wb.add_worksheet("Classified Tickets"); xw.sheets["Classified Tickets"] = ws
        add_table(ws, t, name="ClassifiedTickets", widths={TEXT: 60, "note_clean": 50, "classification_evidence": 40, "review_reason": 30,
                                                          "primary_reason_name": 30, "secondary_reason_names": 30})
        ws.freeze_panes(1, 2)
        # 2 Approved Taxonomy
        ws = wb.add_worksheet("Approved Taxonomy")
        add_table(ws, xl_safe(tax_sheet), name="ApprovedTaxonomy", widths={"definition": 60, "inclusion_rules": 60, "exclusion_rules": 50,
                  "examples": 50, "subcategories": 45, "category_name": 32, "category_id": 30},
                  wrap_cols=("definition", "inclusion_rules", "exclusion_rules", "examples", "subcategories"), fmt=fmt)
        ws.freeze_panes(1, 1)
        # 3 Cluster Analysis
        ws = wb.add_worksheet("Cluster Analysis")
        add_table(ws, xl_safe(cl_sheet), name="ClusterAnalysis", widths={"dominant_theme": 45, "representative_notes": 70, "top_keywords": 35,
                  "supporting_examples": 50, "contradictory_examples": 50, "proposed_category": 28, "approved_category": 28},
                  wrap_cols=("dominant_theme", "representative_notes", "supporting_examples", "contradictory_examples"), fmt=fmt)
        ws.freeze_panes(1, 1)
        # 4 Summary Dashboard
        ws = wb.add_worksheet("Summary Dashboard")
        ws.write(0, 0, "UB disengagement reasons - summary", fmt["h"])
        add_table(ws, kpis, 2, 0, "KPIs", {"measure": 34, "value": 16})
        r = 2 + len(kpis) + 3
        ws.write(r - 1, 0, "Primary reason distribution", fmt["h"]); add_table(ws, prim, r, 0, "PrimaryDist", {"primary_reason_id": 32, "primary_reason_name": 40})
        r2 = r + len(prim) + 3
        ws.write(r2 - 1, 0, "Secondary reason frequency", fmt["h"]); add_table(ws, sec if len(sec) else pd.DataFrame({"secondary_reason_id": ["(none)"], "tickets": [0]}), r2, 0, "SecondaryFreq")
        r3 = r2 + max(len(sec), 1) + 3
        ws.write(r3 - 1, 0, "Explicit vs inferred (non-blank notes)", fmt["h"]); add_table(ws, ctype, r3, 0, "ClassType")
        r4 = r3 + len(ctype) + 3
        ws.write(r4 - 1, 0, "Confidence distribution (non-blank notes)", fmt["h"]); add_table(ws, confd, r4, 0, "ConfDist")
        r5 = r4 + len(confd) + 3
        ws.write(r5 - 1, 0, "Top 10 most common raw notes", fmt["h"]); add_table(ws, xl_safe(top_raw_notes.head(10)), r5, 0, "TopRawNotes", {"raw_note": 70})
        r6 = r5 + 13
        ws.write(r6 - 1, 0, "Operational metrics", fmt["h"]); add_table(ws, ops, r6, 0, "OpsMetrics")
        def bar(title, table_row, n, cat_col, val_col, anchor, horiz=True, h=None):
            ch = wb.add_chart({"type": "bar" if horiz else "column"})
            ch.add_series({"categories": ["Summary Dashboard", table_row + 1, cat_col, table_row + n, cat_col],
                           "values": ["Summary Dashboard", table_row + 1, val_col, table_row + n, val_col],
                           "fill": {"color": "#3B6FB6"}, "gap": 60, "data_labels": {"value": True, "font": {"color": "#555555", "size": 8}}})
            ch.set_title({"name": title, "name_font": {"size": 11, "bold": False}}); ch.set_legend({"none": True})
            ch.set_y_axis({"reverse": True} if horiz else {"major_gridlines": {"visible": True, "line": {"color": "#E5E5E5"}}})
            ch.set_x_axis({"major_gridlines": {"visible": True, "line": {"color": "#E5E5E5"}}} if horiz else {})
            ch.set_size({"width": 620, "height": h or max(260, 22 * n + 80)}); ws.insert_chart(anchor, ch)
        bar("Primary reason (tickets)", r, len(prim), 0, 1, "G3")
        if len(sec): bar("Secondary reasons (tickets)", r2, len(sec), 0, 1, "G33")
        bar("Classification type", r3, len(ctype), 0, 1, "Q3", horiz=False, h=260)
        bar("Confidence (non-blank notes)", r4, len(confd), 0, 1, "Q18", horiz=False, h=280)
        ws.set_column(0, 0, 40); ws.set_column(1, 3, 16)
        # 5 Validation (mirror)
        ws = wb.add_worksheet("Validation")
        ws.write(0, 0, f"Mirror of {VAL_XLSX.name} - reviewers edit that file; this sheet is refreshed each run.", fmt["h"])
        if len(val):
            add_table(ws, xl_safe(val.drop(columns=["note_clean"], errors="ignore")), 2, 0, "ValidationSample",
                      {"note_original": 60, "predicted_evidence": 35, "reviewer_comments": 35}, wrap_cols=("note_original",), fmt=fmt)
        r = 2 + len(val) + 3
        if vm:
            ws.write(r, 0, "Validation metrics (reviewed rows only)", fmt["h"]); add_table(ws, vm["summary"], r + 1, 0, "ValMetrics")
            add_table(ws, vm["per_category"], r + len(vm["summary"]) + 4, 0, "ValPerCategory")
            cm = vm["confusion"].reset_index().rename(columns={"index": "human \\ predicted"})
            add_table(ws, cm, r + len(vm["summary"]) + len(vm["per_category"]) + 7, 0, "ValConfusion")
        # Cluster vs Category
        ws = wb.add_worksheet("Cluster vs Category")
        ws.write(0, 0, "How each cluster's tickets were classified note by note (clusters were a discovery aid, not labels)", fmt["h"])
        add_table(ws, xl_safe(agree), 2, 0, "ClusterAgreement", {"dominant_theme": 50, "mapped_category": 28, "top_classified_category": 28})
        add_table(ws, cluster_x_cat.reset_index().rename(columns={"cluster_id": "cluster"}), len(agree) + 6, 0, "ClusterXCategory")
        # 6 Revisit list, 7 Taxonomy history
        ws = wb.add_worksheet("Revisit list")
        add_table(ws, revisit if len(revisit) else pd.DataFrame({"type": ["(nothing flagged)"], "id": [""], "name / category": [""], "revisit_note": [""]}),
                  name="RevisitList", widths={"revisit_note": 70, "name / category": 35})
        ws = wb.add_worksheet("Taxonomy history")
        add_table(ws, history, name="TaxHistory", widths={"taxonomy_version": 20, "approved_at": 22})
        if changelog is not None:
            ws.write(len(history) + 3, 0, "Change log vs previous approved version", fmt["h"])
            add_table(ws, xl_safe(changelog), len(history) + 4, 0, "ChangeLog", {"change": 30, "item": 32, "detail": 90}, wrap_cols=("detail",), fmt=fmt)
        # 8 Profiling, 9 Run Metadata
        ws = wb.add_worksheet("Profiling")
        add_table(ws, profile, 0, 0, "Profile", {"metric": 42})
        add_table(ws, xl_safe(top_raw_notes), len(profile) + 3, 0, "TopRawNotesAll", {"raw_note": 80})
        add_table(ws, kind_year.reset_index().rename(columns={"Year Disengagement": "year"}), len(profile) + len(top_raw_notes) + 6, 0, "NoteKindByYear")
        ws = wb.add_worksheet("Run Metadata")
        add_table(ws, xl_safe(meta), name="RunMetadata", widths={"item": 30, "value": 120}, wrap_cols=("value",), fmt=fmt)
    logger.info(f"Workbook written: {WB}")
    print(f"Workbook written: {WB}")
except Exception as e:
    RUN["errors"].append(f"Excel export failed: {e}"); logger.error(f"Excel export failed: {e}"); raise
RUN["status"]["phase9"] = "workbook written"

## Checks

These assertions stand in for a test suite, so the notebook stays self-contained. They cover:
* blank detection, text normalization and duplicate-note handling;
* mapping unique notes and clusters back to tickets;
* schema validation of the taxonomy and of classification items;
* primary/secondary consistency and UNKNOWN vs BLANK;
* Excel completeness.

In [ ]:
checks = []
def check(label, cond):
    checks.append((label, "PASS" if cond else "FAIL"))

# preprocessing
check("blank: None / NaN / whitespace / placeholder -> BLANK",
      all(processing_status(clean_note(x)) == "BLANK" for x in [None, np.nan, "   ", "{{FIELD_REASON_FOR_DISENGAGEMENT:}}", "`"]))
check("blank: real text -> VALID_TEXT", processing_status(clean_note("Business reason")) == "VALID_TEXT")
check("normalize: _x000D_, whitespace, curly quotes, html", clean_note("Client’s  <b>note</b>_x000D_\n end &amp; more") == "Client's note end & more")
check("normalize: negations kept", clean_note("Not happy, no longer required") == "Not happy, no longer required")
check("original note column untouched", df[TEXT].equals(load_input(CONFIG)[TEXT]))
# duplicates and mapping
check("identical cleaned notes share one note_key", df[df["processing_status"] == "VALID_TEXT"].groupby("note_clean")["note_key"].nunique().max() == 1)
check("unique notes map back to every non-blank ticket", unique_notes["records"].sum() == (df["processing_status"] == "VALID_TEXT").sum())
check("every non-blank ticket has a cluster id; blanks have none",
      df.loc[df["processing_status"] == "VALID_TEXT", "cluster_id"].notna().all() and df.loc[df["processing_status"] == "BLANK", "cluster_id"].isna().all())
check("record ids unique", df["record_id"].is_unique)
# taxonomy / classification schemas
try:
    jsonschema.validate({"categories": [{"category_id": "bad id"}], "open_questions": []}, CONSOLIDATION_SCHEMA); bad_tax_rejected = False
except jsonschema.ValidationError:
    bad_tax_rejected = True
check("taxonomy schema rejects an invalid category", bad_tax_rejected)
good = {"note_key": unique_notes["note_key"].iloc[0], "primary_reason": "UNKNOWN", "primary_subcategory": None, "secondary_reasons": ["UNKNOWN"],
        "classification_type": "UNKNOWN", "evidence": "", "confidence": 0.5, "review_required": False, "review_reason": None}
it, prob = check_item(good, set(unique_notes["note_key"]))
check("classification: secondary never repeats primary", prob is None and it["secondary_reasons"] == [])
check("classification: unknown category id rejected", check_item(good | {"primary_reason": "MADE_UP"}, set(unique_notes["note_key"]))[1] is not None)
check("classification: low confidence forces review", apply_review_rules(good)[0])
if complete:
    nb = df[df["processing_status"] == "VALID_TEXT"]
    check("every ticket classified", df["primary_reason_id"].notna().all())
    check("all primary ids are approved categories", df["primary_reason_id"].isin(CAT_IDS).all())
    check("UNKNOWN vs BLANK: preprocessing blanks are BLANK, never UNKNOWN", (df.loc[df["processing_status"] == "BLANK", "primary_reason_id"] == "BLANK").all())
    check("primary never listed in its own secondaries",
          not any(p in split_ids(s) for p, s in zip(df["primary_reason_id"], df["secondary_reason_ids"].fillna(""))))
    check("confidence within [0, 1]", df["classification_confidence"].between(0, 1).all())
# excel
if WB.exists():
    xl = pd.ExcelFile(WB)
    need = {"Classified Tickets", "Approved Taxonomy", "Cluster Analysis", "Summary Dashboard", "Validation", "Run Metadata"}
    check("workbook has the required sheets", need <= set(xl.sheet_names))
    ct = pd.read_excel(xl, "Classified Tickets", usecols=["record_id"])
    check("Classified Tickets has one row per ticket", len(ct) == len(df) and ct["record_id"].astype(str).is_unique)
    check("all original columns present", set(load_input(CONFIG).columns) <= set(pd.read_excel(xl, "Classified Tickets", nrows=0).columns))
res_checks = pd.DataFrame(checks, columns=["check", "result"]); display(res_checks)
assert (res_checks["result"] == "PASS").all(), "Some checks failed"
logger.info("Run finished: " + json.dumps(RUN["status"]))